# Deney 06 — Render Pipeline (Notebook 01+02+03 çıktılarını birleştirir)

**Amaç:** Onaylanmış an + script + seslendirmeyi kullanarak dikey (9:16) bir YouTube Shorts videosu üretmek: kaynak aralığı kes, dikey kadraj uygula, seslendirmeyi orijinal sesle miksle (ducking), altyazı yak, kaliteli kodlama ile dışa aktar.

**Çalıştırma sırası:** Notebook 01 → an onayı → Notebook 02 → script onayı → Notebook 03 → seslendirme onayı → **bu notebook**. Alternatif olarak `00_youtube_automation_master.ipynb` tek başına tüm fazları çalıştırabilir.

**Bu notebook hiçbir yere yükleme/yayınlama yapmaz.** Çıktı sadece `results/render/final_short.mp4` olarak yerelde/Colab oturumunda kalır.

**Tek kaynak politikası (düzeltildi):** Bu notebook artık ffmpeg filtergraph'larını, altyazı üretimini, doğrulamayı ve onay kapısı mantığını **kendi içinde tekrar tanımlamaz**. Hepsi `local_pipeline/render_utils.py` içindeki kanonik fonksiyonlardan gelir — master notebook (`00_...`) ve `verify_local.py` de **aynı** fonksiyonları çağırır, yani üç ayrı kopya yoktur. Modül aşağıdaki bootstrap hücresiyle yüklenir (repo → Drive → `PROJECT_REPO_URL` → gömülü hash-doğrulamalı yedek).

**Onay politikası (üretimden sonra, içeriğe bağlı):** `run_render_pipeline()` **onay argümanı almaz** ve asla `approved` üretemez — yeni render her zaman `review_required` (teknik doğrulama başarısızsa `failed`) olur. Onay, videoyu **izledikten sonra** bölüm 5'teki ayrı hücrede, dosyanın SHA-256'sı doğrulanarak verilir. Yeniden render alırsan hash değişir ve önceki onay otomatik geçersizleşir. `render_validation` bir **teknik** kontrol sonucudur, insan onayı değildir.

In [ ]:
!pip -q install imageio-ffmpeg faster-whisper pandas > /dev/null
print('Kurulum tamam.')

In [ ]:
import os, sys, json
from pathlib import Path

if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            import site
            site.addsitedir(str(venv_site_packages))
            break

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RESULTS_DIR = Path('/content/results') if IS_COLAB else PROJECT_ROOT / 'results'
RENDER_DIR = RESULTS_DIR / 'render'
RENDER_DIR.mkdir(parents=True, exist_ok=True)

STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}
print('Sonuc dizini:', RESULTS_DIR)


## `local_pipeline` kanonik modüllerini yükle (tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['render_utils', 'video_analysis', 'approval']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'render_utils': {
        'sha256': 'e6c7ff3f7c47b499702730b8407fe7947c13fd460942d044e326740418de8a7e',
        'encoding': 'zlib+b85',
        'data': (
            'c-rlK+m_olmgswZ1yWfvloFQe;!E6eD_M>&9iMjWESHl^uW}uklBg0}6shK=%5sg?nm;%%^Eu~h<_FF%oP7a600dQK$2~pM)9#K<'
            'k-)}nV`Ja3zrX)FuFGhW#^Twt)jGZqMV!sz;$WSu<21=)v53=kT$G|*L`6IklZ`mKKjuPXRcMf`H`zq=t`^C3v6p1{t`t!w;=k0%'
            'Wt7HQ6^h4s8chxosJO=3StX|VdL#0=s1`{n()=Qs_My^DT*gH*-;C3I8l{8vW{=^DWVOzVs^sFUd{#rJXGsxHt8^pgc>y0{L6zhn'
            'M7heBv8dv*>JRqzUgl6UyMPfDv8dN+9?ePt)z<YSP09uJctb@7;&~<V48RsK7Gj|DB&9MlaaAM$bkBTUQ-hI!Hz`bCS+Cb=0`ryE'
            ')w-_4I;s|fz5V_Dy?K$Z#CSZftGbBCV>v1So8?thC3#lv?a6nGs9dDUME(48nP=*`h}BcMsH-H^zv{`l$ft2xsyEdF#{h%kmS<5F'
            'SIH_??W*6Qz`sA_S<L0I!_c-W_7nWv+dF;p=*`ck<6j=V`1$Fn7>VmW0e|-6_kbZ#JFB8}KNS13_%crOHNF_H(`Xtm@)TwVKNaz1'
            '5?=xS0Ac_-?S~8@T0^fd@weDdva+7flWBt0p?eX&oJTOr^cf1y>M2gt7zefi5L@Wk&)>g!{^rHg@vlETKmF<H>+z4To;-a)eGIB3'
            '+XSHq$`z~)KCI$dQm^nSjf#u-;PTD^EXE`c0MO${KfQVW>gD*&4^Mx5I{xYPt7p$&7-J1)@jR;2Dj3Qa^b-1nSTF{}EumxZ@i=V0'
            '&T|+@7Oi4<A3TZw9Q{&@(<mzgw@jIQz%t*Q7#|`S>Sa8h=4oDxqq0O8|NO%d{CNg{Oc0Z(^ptz{%q_Qy-jA>HVunTTI`5~j$?^T2'
            'BkMzPk!0gbcz<$YegGVgizze<U!Fy285`fyxT=6cl<Q~;xCaGf#V!96T)=#c^Z6Li6L7A)X+BT7=JUs>n8s;Zm*Vts=Y$^p%_sEa'
            '!N*SMo1)%5p92$QRRBUwi)2k31LkkFMjY+w5>)C*6tD73yn*(kViRYfc)f_?Y*R$$yI+zrE;gn4>c>qP6-fj;#s>I9UYD_1@K>O<'
            'agmpycv3_cK+Gyw-BA1y`usL1#S5UG@YsU$yufY|DnR>Dx`D>9Coyc$4IqFn-FKH5@uY<dr;iU_#}_qJ9#qx5jTVdO<lcR(adhYC'
            '?%il||K7>TgUQjodv~Y5i|$R2<NJ4J^ON7r?#A~XMBf}gxc5zba(wdO=s1ez-`syNo6nAq?oVd(qZa78ND=O$s@CQ3@UV!k1{X=S'
            's3$N_#S}^aQ5#J2)!{{+U!?INRw@ryQIZ|z^AsNB2+HZ<Ek=_j(>MbpH7DWKvloNvedR0m89V1`WdcVa&}g8_X+&ELj}UARHct$2'
            '6f_Cs;C4e)P}LYd0=nHtNqD{#e~i~~+%kLe9uDbgf}F@|4G0bg>|v3o=_HyiMNWHZARg6KzJfNV;t#Lhhyo6DWFoTsO2UK5D~a`3'
            '<-k|K0bC_j8Xw@{M0}M=p@gJtBAUQ)H}I0eF>LS;ar((8%ww+JK@0Cc_}kOMZ$39IJRY3vq6<7D2frW*gB7`fCd}{79^9GTxjQ?a'
            '-=BVS_xSjCcjCLzbbfaNL}C8Bdv}iGd(r)GCikO==*H~mDEekL`Q{|L8z0T@-2L1%fzjcR2k}q$zQ4LyRR47E$={Y1$WDDS8L$S`'
            'xiTNO`MrJG5_|ZEN5IUzdIntG>sK$Io=WcSk5B*f>#NsKkh|+OTahIa=1Sn8TF?<k??Bw3Lk(Y7>deC*SMt=8Z3YP*XAA>B&zhr<'
            'e$pw7PlY^v2}S~QUd&@;QTO_LdwalX0-ImvDelmS4|eFoD()foj4jUi_fV8o0b>Xvpi}^gR_}c^#=rEppf3)-1#xLw4VmRe$&Y}8'
            '&@zjm=~a>?C6YPWn5I=E4`@<JO`_POkOYMqgs6K3%|WAXFR=>uus?v$$-37cr1@1`^!ijinHvzkP@@@YWU#)2tR#|5_?h*g#ak2+'
            'urtV0)bZ1z$P1e#!awu<?-P(%S23)n9=x2{Yu)A-#rC1t_f%fRms6DRXtqH1gGwccO!2Je?_PgvL+R^Za<yk6ik)y4!9?oj0YlUh'
            '@O&~)6a&}Yj5o^*ka2*}&&Co7<XV_0K?@PslaGJkjGHcK^)e`BPg8FhZ;-5$n5~{+0dCk=ux3W#cJB;s4*$GF_HVB}F91e%&RN>_'
            'e6{3?P|O3_hvBt^vTgWDd;YqBDxnV+A<KFC#}eq6U<PsrfEbyX$F{QvfMhibYL?<AjY6^yPMKPXy(O^#{1CRd1o(s-y#xQw6!ddB'
            '{aZ<F<QYgO$ar3-X%ERP(G9>Mo~2oG0lkMgk-CWi*O6#+1t$pbO>sswk-$xW?f|c~h>(ys6OCvzUDVkU@gf1@(@Ue(WEKr&$pLat'
            'z5Dm>-MQZn#RSmRUL@|upkAZ;qDPSW<`T&7i}*c1^%OycV+7SM7Cjx~@zLW)m>XDVo<E?Z5unR>mK1H``D3(1%Bg82fDqATl%&M-'
            'k!C2}7}_itWhvH4mcb4?{o&C8Oxj?NfPK%0oZzzz<`8(!=b0#hwE;Ggpp)thRbDZjLh*ZG+~7z;B}tTuKhOzHu%6*TNE#`Tlwy`&'
            'WvJysHB>f*S6P)4fF}hUq6f(3!Y8RKqp<?F%am^>XOljm@&%P(^$ay)u(C1g+9K>Ds`lVz2`fJ(<r=gA=Sec3#|5hRa9###&gfq?'
            'V<TbC_?gDtb1J2{7$_z`3@+lT=kRv=<iPVly&W3#l=maok~{+u8RpDNT-xSUnb*LnBU@2!V27><FHk3hurw+_k5C!bYjDH<K3FZ`'
            'Z`c4h{&GaDE36nCt>b)2Ke@sT&@#zL;|Cxc@P~RJOyJ+I<0%pl-e}EyoLPM9xqFrx;0H-bQ`TecYI~a-W%aQjMkBZF8zvUd855E-'
            '$#e|qIkTplc1VK90)AIB4&(eYG(Rr#yi$`s=3bBc14+byiQt)*|4>cEHe$zk!Wuw_&HVW})pIq2CX^ee&@6nrIb^-<0CnIeR`t+a'
            'yXFvdc;U`*@jltjIfuvqn8&MAycw=dcD;R;bwUe=t<JPWldus@`b|F=A@ICapfzF;ZS-+GQaugQ3?4@EaNX0NVHd-9;M3hL77VUc'
            'Z?*$PCmXh)DmM1W>14B88Ng3I>#Pllg4H=|)APC--MvQ#AfW3SL;#1^u(~)PtHOA)sp7Jy%JV^JOp;u%HU?@q4f=ht1O06F_;sCO'
            'hnYbRk_=`EXNQk9d0fG9a#7T9PDf=TCJ`zG%j82~cdZ#s5tHh_=S61k9N&7LO&Qjh>XV}zrkXg}PsGyOuTK1aiW;V;^bb;Z=#zpC'
            '-`d?;7jQ85=E2W@di47Fkmo#2$|x(v5_TF+w^|XjT!4IGfZI9?&}<^bAB+`x>+NK)zn;Ff`UvROEGwO4rArv>rLWNdxa{@!Za_4T'
            'F!d&0MKl97aDpF_B8^}vqFEw(*Y-A}6*}McVK#6U1|HFD|5;Ql(}ci9qKr#GS8)+ffiv1f>m)_E14kB9!tQi$uXCq)kIP+q%`^0Q'
            'V2l){47JIY@TLGxA1$QVS+(RwqiV@N?<vd=TSxIQpQBNDJOOY;66<v*?E$u4ie6I$WUV=@(j>+o{ek%l1h|cu#KdZ8hAoFtivRex'
            'f9I)QMHg|B%NE)CS}u|`DQe|Q>c%GI+6+8|jtfLwXaPCUm`KPTOT&U0IaNK$$_m(Wt6zW&(|om}=CE@E@t7>QrC8M<=+tGbm=UOe'
            'YZ1X>OM~<rN|u6@#lZFg1A;uecqdysNZ$1=ZW#`-dWD8qLUp?jY=hUj6nU)KbFwEZs_91Ao)2W|4lu{`2_!C%uI#Ev;nZ5j?_;?&'
            '%|5hY(Kj2_I+j|=Rwam4%wYiQT;PsD$*wI#!YH3oSy(l)p7SaZyM~fsyqd%_v=iZqB_Q-P7{D+uifFw+cxpC#K*I-G_g;>EkFdT4'
            '6nva#K=mqKr814k%nyqN&0IyrOsH4L)p7qz0vL&3f#&8{AViU@#42Zj8t_c?9uILmkKeu>_Vtn(+7@O4)d3j1eTzrQVI!@;`O+ix'
            'hxk?;Q`7kC?D!lR#$aeDy(Ccjd~`0p765D*z`yk7WO#0_4!)C1F{zW3=zW!dAUZZ?t%p)1(SZ<l%B)%pQ7jC_B2F$AmHaVD>tb9c'
            '7psU~Ko9V2b(T;O0RQ65i|7I!07MVAVagr|hZRu56xVf9pzI0D!kRV$nL~GRcTN`(P!HTP=;i=$R<J&2N5ji=7GNWx#wC7z|ATU{'
            '00B8hDf)~m%+6Wp<iQ4TTB5;VTIB11)L%Hj0w^_N#YKU1D6pkeR5_uzjSFJdZplFQAvWKqjsha3@#vc7cRRf1x!n!{>89~GFOmzO'
            'V<|ATYhvG#P>+&qit3y=+;;$Cvn^0+l>3gh%<Q&jjJNFlI!%C^I-6XK%VK(dHs^mIx;;|`{_3L*FK{d)Ldel|gPdFFfu~z@U$+iU'
            'c6MkMbBCa^cc3^6=ZK+cv>Ek&J-F&0o(y|G94z49uEl6+p6C0MM$s&FD~zGn_|F%`R`e$E639cG4TXVryd&+!?C<xj%^uY=B=^Qn'
            'Uqq!bTKqf|lRQsBz^tcB*oI@^bK%sP$=6<fxly@69iA*;QGuOAGJ!_HOJG(|Q!d}Z0ZoN6`@^SFk5%#>=>_0rbS`>oG-8@2Ye7Te'
            'UQ1ssQaVk63cx;SkG{{+#n~~z(jDzHSviY<lAI3=0)SkGXAa#x1o(jvdAOr%cLKHySVMK^{uW!Z0dCAVB0oHNuyuED;ElhU{1B<_'
            'pCLapi_h@^mchF%u!1g#e-ZIUYdSEH<p_PxN_74j9Sx4|4SB0wH<+{?MpXqEJ%VpQy<u>pJ4Z)nSOE4@yFvLu^(cbBfRq6FFGnZC'
            'S<NFI!4Zu*-{A~)4ov-+d}9*&bOO+g<uLZDJNX|q@t-j1*7VW5|JK21n_-<vR?lF~<%u5^7v&Iz%jyh%gY5iOh^*}et(I+mQ5;BK'
            'p$I9vGcMz)+|lA+nVBEaKfJ%S^WoqzpaAep@$4sl(d_#gHoe^Ov>gvN;$Q)o1Eej>fFK^EsLj5NQxPN41YQR_JrJ*vnwFLn)z}<R'
            'sf{}Q8eLS&3J7Z<dKJvXG=e_#k<G~R?BGLO<fx0Sr0V_G1rXxzU%mN3yi>FAPV|5z<&!clfZ8+k!4wE?1#&*|YVX=p@(%jQOw0u^'
            'i^LogCl370RVLmYG{9lj<7kB(KdKIx?S%a@i)RM_C(mZ(L75-SBVgc5iI{6LCW`1HLivS;y37!+<a|^|2I38}OaM9p!7-~*7Fxlc'
            'f;mA_8cLa{BF7gH5LYlYL$wqj7*#O4i~I`Y9uRFL?1U&F`MXv)F5*?812hFpc_sT5XBRM2&^Y&P63@{ahSw!c)qKNjLIpT(P|b-S'
            'HAR=^4xkq7EXzU>j2qSvgr^HM$bqB>h*50RNMbm)mINQFS1~lYh%V!j?IUP?DP=!eyq#JQY=y8G5DllMy|nnn^Y-^6anz9R=?cxJ'
            'XQp@*9FWhciQ4pAVK@FD1c!&O5W~;+8hubcMM<LBO@PQfXd_fOKIU1M#s%d24Z}h+#OZ*^8sZl!D2|3VX=LVslC+h64@4{2FXGw('
            'M~WAE$Gh$900{Td@Z3~KNt%kKqz&aaO_SO8KyN85fNto+W2%?sh{>frskDC3GSRJGdo{VsYg97Y^Zpa*9ckuan5bhkKR*BIX()+J'
            '>-}zCO0O93Q1u)=vqr~#=}so87n<*2fikk)k?|0>6+3eeCFOJ|WkQ>ut?cPWf{Rz{YJ<&GC~Qmd3-Ae03CO?->cm-IUo1!{PGBeD'
            'uEYtYy*`81(j4_0Qb_w!P6UxJppPJQ<yQk~0b}|*hTa%w46gY3o-}#Q#h6BA>JM9bU=z?FT4S_8j}}Mbn$)^VKZZ_S<`8?E<$nPg'
            '_WKu4kB*M5M%GbT%3jel7i5r>uJVI7^wdKqpST)vpMh;A#~ss#$5o+UQBg^yiZvY6W>6GGk_kNom_BE~ix~JC>;bd}98u9z*o`K1'
            '+@<4WK86KDKf-th$W2GCtE81++L~GU!BNOAW5A1XC@xVzfwn1-LwV+`VL-%X^ny{)Y6+?;7o1)|TnSK+aZQ_1|6G-UM@x*JA4yin'
            'H=6=h35Y(W-nP_Ducp0`u)3RT-h8Q!JVgFd1TPOC1?F)zPYO6?Nu;x&eJS1<Lwko?jeE$)HqM0vJd4XU9!Cfb)zlB^ymNrT6qbPL'
            'MX{lBCEB{{EUBbEGsd1vXHR&<C$g6hY&QsIz*exKh^mEZ(hkM(80aznH#o8<O`(=#YDkvs&By|N+z$o+meBA8c4`!qS+J9)tpH+l'
            '5s&793?sTmki)^ze0z9J(CAr|nONqt6n0tbprBJ~B)7>`l&B3QSNVJ3z?;3@9F|m^-Ph0;q~A?f49Lo4bf-l=3_|{f3q5_m#!wX>'
            'tR)akXIHhluTNU(*N`D`cKgp%VK7Yros4>lzWh1I6_$Ozr&d&+6eEjZqM8tCf$@q=ra&qiH@gm|;G-A<hZpd(0bXP?kn1C{t&L?3'
            'YVAfd$&2ieQ23Z{Muj$Vb-XLlO6k0ri!XbI4qOUYTG{P*TC%bOO$2bDzmxR@vb$Fbn*24#^SgQ{{@3ZNmvnw`=omush(NfWjwxXt'
            'rR7G*a5!R)*?;Fh{_WpobXrNuRhFJ2^jp#%?r70a_6%P51C)Gc+PDE-tkW5C=C(<MaSP-Vlxp%hHAL<}7^|m(^jzhNIeoj3P#GhD'
            'i%Dy?2M!tp^YF?V*qPA;vWJ%@+Z|Zb-@M=zpx^l%|G~gJ!Ud_?KW~t8H$xt>a0e4H3hHWp@BpAt>*Xkr<^hT5{k>bU?HxN*&W_rm'
            'E^B^b?Mb;6a8Lt^f$XWmFbIX|JZqOpq^w2?Guk=@Kgxd(BIZFnpQ9Oza74&XZajuE!S<SWa=V3u_tKunVlY_)P`XoxEL(h_gN~)E'
            'PpmGQ{$gkcP6VN+hm40(67{rAJT+955^a=TMVpee1!bc=z)e(1F<aSt2_OZPBFl^t<*PRWk&49jlC~e<$UwY8Eu|LD;knLqlnD96'
            '^|G8D$e&2*WEe2}h=6fStUZw7wb;kB&p1)aacQ+Gtp*0|uAviYLe7f7+pS<N^o$URvE3!E1GOE7;_Ny=6r7H$&6@qRKuF8kSA9mt'
            '<|M2BRXWeHs@{0F4aHzEP+KLCn+i}fP@)%`qq*>wioHQ<fw90Sny(8q)yHf*g+6J!tz3KR#38q<C@bPa5nsgb`LIP>{-Rz*Srd)J'
            'b~nTkXr~D@)$zlNs63!?9^w;ed=K}uUobh~9;yK_V=eU*fY`vICg_kV!=r(XcJ&TS_4Ee1f#~tUt7+cDhCK5;%p!JVh*rraFK$l~'
            'P4U|myps+>_oa#tCvTOK7;BcBRv@!rm@b|fQa}?w{_z(X7d<C09&!>TZr1>)%(a4zwt~C_GT0&MdJ#?Hs(gr|ud>6wd#B0jyLTc-'
            'v-*|ds+>S94r>yRUY<S|L`2s>Aw-W%%kLvR_QiC4d~)wDEA()Kvp8!2V_A@-<XIB4hXL!TXxE~C8{?f3rO+oRL>uQNosi6xk=$*-'
            '!-iH7YW@Mx0<gJS0Ky=mi4?V1yzLP22WqDE!~&-N{N~xggNFt@d3=zO1Ti_9qyB3e)g=oav>^&MjAU^P0?MP)$IqWb9|{l!%fVmi'
            '1k*gJaR9NPE`@ESb|JTU3UC;daa2qfy`rIdZ_BTGZ)ackht1!;?}l#&`0KmA!ImLNnnW+WQzn;RLbW0MclN*EX6Ijh_Y0OE@(%|T'
            '+t6+g>Jb@)TA&(Jk2P4OYb2*|qtoVlvj_71?f7$-H00$b+9V=h)*B)o28(`tPq71X$(a2lJ&o+S(LK{ZReI;9_ucQmcsuLEDqWw1'
            '_qP4_`l<IFet?-V;0@4xx$YeqUUNjr0Vuk`=|*I^<9?rI7S%QhGdM50x;z5>^-pfDFDF2$nr=Mx=wI{5lV)no!x@V>ylpE~C)oC#'
            'ZNkx8y1Ol;YTyH7UX-W+$m%MF2c?%7IW>4*N>j?T8$;@kGf4pZ;wy3I{?U>6nr2r*$G7+K9jj(!Q+?Ta?U&ZD67nhR;seYsY_}E<'
            '$^u4<>)auFRW(rfGmhXu4W#Btf5IAYl3>Yqa9kiURy@TOnu2ryN7x)NwwD*5ux(GTf#$qm?&kTJTc-$%Y}QBUMXY`*(~kDBA_D2~'
            '*@4&M#q=DofQ(j3DXYd1Rp^6Rp?jJ+n7%gKXd;0uDe<xqAWT_0PHB==hJzT%68XVpXn2tX2*(nlwwb;FQ+aU_7Z!X|(DMp;E!~WE'
            'FzXbs<pPJzv~_EKTbf={`$eSPE|j*LI&8|gdUi;uy=R*+Zj^@1{@E)j(b?q=Z;8@j<5z0>Yw4G!<Cb4T2b<Pwzy-;2a1SU(^uFUW'
            'Zv4TKN1aiVgi*#AgaoWdmQ)+X3oA2aBpqCEUeU$cl}t$5ig+L33dOTPI;TV;U|0c{geK7a8OU&ro*(!Ldn5}3W(PB1l#;Y$gn<1w'
            'O>6QVWG_u69P{40ikM#?-mv}pQlKJ{gt1>oJF!g`#fK^K*GxND$i>YMJSXZ%kk`kPSjN^$IT1G<23+Lm>!|jv+Mdo)PP-H^l{it>'
            'O(SV)Y@obN4m(8?DWy`>DD<0hwD6RvDTe~|)|OOw{y7Q`Q{QZ*`G%&sc^yc6<*Ai{R9i_R=|rcX9oiRr2H>-RoT=HBJU|PP9b?lj'
            '3i3BG)SkjL9(skAza9n`Q?3K?tvK2<<t}gAh7FfKtB}HLJ3k4!0WlB5Q?3)<@Fza-oliCdBYtz57pM<2U>MskE0akY8{c@2mHgU<'
            'zU{ZevgGPEK;(uR%29^Ro*AlTl~Gp~Wkzh)rx-K!o)g)kT#^hr(|YC(1J@Ddi-rl|UYt2Nh2`Y)-mL-_cC<&6(JI(l+TBI1rS2~3'
            'S?=z_?d5Nmb>@e4nCFsRgBkd|SkTA>hyolR)96Xh&P~b7kRM7?qmU%8-ooZ9xxT^gQe>QDtj`U}>ytq-_k6y0<HDlrwbkuD(TN*Z'
            '8S7bn>|6B30@)zrRVa`OT3^qO&Up|lPV~c$&r-VRw=tl>E$Z4lxW3xHzg`$o?Kfay@5+iyWbq5h=)!u>J?ML)?rw$UmLOvaG4oYr'
            'A(qy+UpFEYl;MwBNZ92uV6?&#2~=;kRdC+2^7Y;T?hr>E8s_7ECsf^cy$B}9M1xi@_HN$U8ZfzQRE_lSa%$*iqzl8d%c{vhS=#3S'
            'H^M+vV2gT}ecC;jxO?QeM&|QU#j3Sw1V7xGNbG$n4nE%>fOe{sffsQW%Mtf(k#kb<POAu0u<U)Q{lWr}oPK^ee*N_2lc%r68g_LI'
            'Jl*H*HX8*AeaB_N&H=!Mtky~-_If`zk~yqr4>;_wB}W?qeGy~Qof)}@02jQ<XR2IC3_2G!#s0D>sVj#dud9-2nv_agLiX1lRXU(I'
            '_}EAJ{`lzV$i@~RG>u+hcyuz`4zEq6fboi;(C_e?8yp_p!Mq!$$NCfya*R59KxOjwE_>=h`VS$adCn3FC67|kD0fe2Mw>EPG-&{w'
            'q5dH%fUX3{NQ43EGy`ds5WP9!%K?Z8<r;bWY6F7uK%AoUvkU>sHbR0E<b^B4xO8vWZ+um!Rk9{6_9{lVG?Wbhdr?+_0cuege>6Ha'
            'I@~BEP3SJWQ9tnEjC*~q{BleOKk`?^6hC`A{cEgvBS}MeGVOZUSFM}knHzE!_{TX~PKnE>m&BZ59whAFkUFyM@9qiZ1(VTC1M)IA'
            '1_VII2EoOh(X<1&Y4-#KSChr8vt`Ktj8pRAJHs+|fiYE9(<&kD*>29ftlYYuG;~bZGX_=GD@;`&iym3NUMkfM4b^q>f=;Lz#_#?1'
            'XX^de%xzE+@wGU1C)I3!XIXLyQ&7L{TzR&XMQAf}pdXsk;_)g&*en>Ba|->i7V|5D-mdONIAf_azUhV<bPGhlPBEPN#lgJxnrjgL'
            'a=Va6&Ejpgi?8>--Qw@QB(n%w(n*dvhm)m;FKwcz)t!xvfbGH-jY($jLxhM(8E3Gz1trX>>8QYdj*bnkhXANyksf&V-q6!qpXdA6'
            '-jfetYgFiO;kf-)>?lYR)8FoQG(u&V1a_2A9-xh;K}PChtzPgplQT_~`fYQTa(GY)R3|aTuIf<ICQe8S1+0@(IvOX*nsHW**BHE2'
            '$IY2_gHy1OXVDmA!BWy;Xuv9UaT;Y8b#x&=SP>XEK~J)z!fXJX8`j7Rr_atedJ0P68#U|foP(2S&w*Z5J~`9mBEzr?I3DS!Z%n<k'
            'Z^WyNPFLv|A$n2=EmUgo>4ryxLPd?)Iw{gi#MN}rccQN89HESy){@ZKtjyv1L_RRoW3cf`5uvwaDLpmih=Y>#9J7Y?23C(k4jLsm'
            'Y#-?$QYgK@+*ER;V~(dc;)5-}VDwDjnH;N`lri7G^3NX;37SMEgcGn{=GR^uVK8^hWipLN!E{|yD!|pc#?1iY&M1I6c@S7fE-@Zs'
            'Ou!Y2G3hQx+|;1i4|YHrQUxD!z}BL4Xc`k#>Z8kOCZiHZb{bE4oXyD1cuZM~q3?NJ(gKd8(n4YUNIisN0&+5qc685%W~2D1Xa7bu'
            'FZoUTQp~BjK<a6*KLdxB53EzD91z=MXe!JZJ$IS4Hlx?+`eE;}NnSF2T4I7{^!f$NoYn16iyQfB4}Btwh1K2x?O!Y(+)n4_CNQyk'
            'zeqkrZq^vlqpThMhxYc=n(t+u!g?oZG>w)6W1JmIM>0rvB&_Jx8Z>Ckdd~4jDmuN}preJBhQe;Vtq5_oABerjmNWNmtCzQF1khr~'
            'dp6;e*&!u=jz7<W9;X&ZkCZGCn|#?fSZB%fpvrwD?pxr;lFY)UqSnrmVFK)7Q{$YO6x=%n26^?4Le0?PY2B3U8}U(yxRg9_<QwLp'
            'X5Wo2I9$Ax#jF`?wtWITt8PSZ{vLMyxbJ-H;OOkgRUBj3Bs^>wF|&ud%DKC(`D6*aTP5)GFSAV+t&*uqbXfLo!@S(UL8vsLpUmLg'
            'md&{pnBTw(-IgI@lO+pv&ABGln9Cd9v5}l`30GK7xy~TV`YMbjQkKM+K_M5<X+=uus&+u{7s<sUMU^YvK=R6JAObdtu|e7qqq3E^'
            'S5r^2M-oU)$tj#N(=GwFx$p0YHTk<_NX}a{+ju24NxaAo3*uz&lb_mht5QGGcjkHen(o;j(VgXn{iWF>S;eCpGIb>yG!G6BoO5q_'
            'F`Zur3sPx)SUK7;kCsfx7I}1L+%LV2rRTQgZ+Bzd8e<)Ch|D+eTs7_VL-Q5Rov)8r_xB@)<ZoSSS*Yt@3pm9ev@_@7?CGWTLecBZ'
            'YG2v&JIzw_d*X5LbgZ(e@RQc*2(y}08(rTRvGEzF8u^z{-1UIo_Wu*PWqJI661VJj)1&E=r#7mQ<jL*u*Al*QhC5XWXjg3#xPK#y'
            'jz`NX0X{oU@h+&{Ec&eeAWmPu5jS$@QUX>P2s>3O)@*e$CZ5^1@uq#oj6BEAo{woTh_^$d?Vmo%+*)29uu7sGmE|q%KvKC1i<mej'
            'q0lhS*rp*i%=B+G5w&5nwWwt;^3=7AMydi@bVA{Dwq4;^m2YfD)XDAeS~e-IL<o-_2iGQ0dKH*+$ywQYnpL&uP{sAZsX)qlH|yH}'
            '`@jDuFkW8cyPo-Gi*Neoo40f|gV35m@|V>a!)a@>e9|^j$=8w}?mi}42CX|syZrD@%&VT<Ka(Pkmg9?P?Yj2ezt<SanvOQ6vOnOs'
            'l&(&47G%*X7kNRhv+N1GUcjN)1kfn<b{gS%qgY6GD#jgAP~LCWq8p0n4$BB?*SU(D0|59i@|Kk6z80{~fU#LAr7(+Dm^=_I-3$l~'
            '5F~D@Lg)S6&H`MPHPPZgzLdbuj?Noxwgm=e+<~BqwO||^pDXxID7NG;;aiC=0u^SjLUV;#QCa{C6`GHQQ4`zi1p~y?FYvGL@UJ2M'
            '^)PVq(Q?S<nD9j3bYc-j%yGdj_NASucg!S6BXz&cxL|4ntZA(jy2W7q2TgrKbbo!l6yHiy_&vohxtqaXAJ^bpWkx?%b*&xYujrro'
            'j%zt2o|^O6=F)frG<FM~PLI{A*&ln(%eHaQw>x$;B#9xHuuMquq9+oFfT;TD7oWhG4=Rl|n3aG0UQwsxqX$R3cz#`Dqig)`1iczm'
            'qFyar`)@yN|Is&ADS+RcAc5qoqI6EmlbyZt5M(dm-2uYAA4`wJdr6juHe0u5S+!eNXSM(68?ZP(pK`3)RmHv;%RCJ{Zrr~jYKTXt'
            'r|40Pq9#WvP>2D=jx%my9^Q4nR4N6`o*bu?^@RXEjWDPN0ia)W98prwm=6)_upn0*&gH}w7rrN!a;nya$mf!i<=jmOgG31Sz>Y5S'
            'WX9F@)#jCF7LeUdqdSRb`y!st<0;3t=krEWvy6&}?t_gkIN|FBt6;?AX#KdZ4k$`?l~WX-3?ZZ*Sh^D4T$rk*i!Cr%fsuYPP;b-_'
            'MbkZ*0lUFoM)t_yiDcSw#S&Ow#l3$3&OH;eXmx=M{3c(j$gTzF=rI+Y!PzORa-V1O{M@{2i{GPP+)!NJ9UL8fZG4>vKfxY<9hd(y'
            '6y)I35Q+ZQSME=4xs6p0T8q+n_B~8ZTs+EVr;BKoUkw4XtXolkEu!@)dgctpiTU1qKKteF*MbU_)_9(gGJPmsMtHd`UW}6IC)P*('
            'R3t!BHjfGaLUBqQsPW=eT_J+1pN*6jd=D~;di{N#g1Gz~*zIH*il31U###Um&g*3iK;RKa`!Dj}Kf%AeKSS{d14?bsG$k}1-2<pV'
            '^8@9T*calpf<O&lsE60;;g?W6l{Ri`VdOkMBE<d`S_!W;ecpz&Seokg;e}1R;Dt*Jw_#v_XNrh_!hinZn+Njo;5a;@|AAaU|6TJD'
            'ZPTg#@M4Q)!h7)Fi3Ju89{4}InMv{wojdMChNncyyom53+MtAT%36=7sN|5-%qX&JLW!V+;!UjYmE+S}5>CtIf(Zl4L+?ZWVe9;P'
            'qxbD3QL?ykPJ1f3w{b}47M@b0Y6r)f%()pfV~F1q^X3D~mhSh}m<BhtTcHYP8cY{+41{TJPK4WJ`O$5bXAZdYZq~egsaFSlD-2Z8'
            'Vz-##q}vAirFcnB0f5k=y2@?o2#z+2GZ`q|xRQbZ_b=fn_dvrm`J85WElN7j(#=ETdZj;O$eod`3VX{~=m7?Dd44Gm+j)cxk2*RJ'
            'IIF~TTCeJqS@1H36CUr>!@H&idwvZhW}5P#o0OecGa0t0@Blv?f1>Wwt9kV=v8jaZ?x3Tr<0!SUN40Lsu4+TyZfj}-sH3&k&P)Sh'
            '?s^oPtob~SO>UKKtDK{`q!A-aN1|Kw&U7;h^zx?%|AgC9q9lz@u5%t-U%!1jZJ!VS{PykSc)PvU-Q5N^0&z|q3Bq5mJxV`TbdO2*'
            '%%LBIx~}5E2H~mEJyrAlCrOm%7icVn)6CTolBHnVdXMIUZRopl(m2HB`5Im~o%Y@@d6R#f<Q2b`L+R691E>jWJIwj7I+Xk{=J^VS'
            'CZ$K~0`<-OCR^#a@0WrY^b3U0y7Je{N7(ln8W*>CE}>X_^d*M#+uUC`zv=42Mfa_R3-iLlv*GdmbLTb|83Mry=iVZ(#;csIeV8Xi'
            '-C9`2DF^WINBx>=$&e_zqyrC{BALZL+xJfiM)S&hPB1qSDhA_t$tpsXEeE$NC1W32*J&%jp^hB_apuZsI^>s87NwgqDaQ!H{J_^~'
            'lYN5{pYCaOS$xG8vNiGc(?wp!nXGL@B&{fk8}?aw<kGX!DuwD6K9YOrE*}+f{|v_aGOwPY{#|ET%g9_fms!=M&a32`{e%qrnI+{?'
            '92{`gvlSoJAYV??I4YRA%PH$Sr%(x{Lt!K%i_Pq7%uPm<BB2b{hWmk5xytp#EHQ}64l>A?J-*$C+<0UZue}g>HZ^d#l3hY>`1Vry'
            's<ultSw>(9D5h!TBB<Hbu`Vau1&qH%Em=zSm37+V8{2FZbO7AYxKm74rkt;M8;MG-3M8Ttz&!&Eo5=x+(Hc{z=D?tqzOIMuaT7SV'
            'm!Ug63HN;H+BTDtZcNI6*3d{o46d)Fnzi4*;hyFBK1kF_nxLvB62Udi$u<D)Ws{Uq3R~|YNr&Q^Kz~te2PO^=Fg+cd10U+8K!z0H'
            '6oY+BVbW8p^Q|#|JM<^>eS2cjz{yYBzGy8<1>A7&TL0c+&ae%1hI;Mx#Ko6yJxj5(XD)!=wetqrZ?8D_vGu;)SJm0>BY^Z_#3OW<'
            '>HDsQ+(msG3gLl9sg1AHosO8I$0kg=L-zYm>4basI1PJ9n{K@Q?Nd77JLha#X>`X+YRt1Q=o_nL7e2EKo`2Gk?FLCsAN0a^A1C8>'
            '>j1S`;^1YQAISXNK!yo4khjRQ<OAM+D3-vmC3MnODj_IPhgmWZwv-U;Kz7_1<ucBaED@MeTSS`(MD9UgnERV-xy}TQ7=Mh+PF*a9'
            'a@oO6si`=|vZ<FwY`11ZD55(rl0Vb62+L@bMN63|&fN+?JdL&7iZkhYVUVo@P2<XQbFX;@W#i-RhS#D!t-}R|+bYyCZ1?6H>`?MS'
            'E_ly}9Umejf!8fIgo+r4`nC-4I>lvD;I)rQ2{Y&1EfY91sm$UY>Uz+k&1z~JY0KUeyJnEWe?NWB=0SUJ^vn&bCJA^)++fWd075RA'
            'qyN*ZKf0(CSed*&mBTjsYFtiZ->0Pgf^=ke&&X-!mvH#qm?;&d`sep%dFrj<q-xjDgCT)xmK6F~-;XE!cc?Wm`E~#!E0Z#nsxL-r'
            'CS?=dvuyvcM%<jV@3Dibo{+gTRWz6IHT88N*DCtEqtN_#3ukRz`XfBEbsgIcn^JL9d0;0<=EQDfeo%!Bdo)`UJyqy_cn7Zj=-8G!'
            '{J*Wz%~ak{+Hdx~p_0$mdqWk+LT#6}N*Olm8)2I0&{6m{ssQI=C|^=@avP)D3na^+eE)d2u_gtxtG;^u{14AxK6=5<e#E_KZ$LB8'
            'R#GbQ8b}&m@Q8k82Z}B<7ldk)?o6uHNC#04#B)}I&+?d+_LTk{DCCP5KcWcy=FKUN3ddax$RXeDIbAG=cf!^c-vklL^wV*32@4M*'
            'DGdiltZ(MC)v*HXx6gO4%cOIAraqDy<gVzekQlzq^~}C?QitFAY}=+eFByj~$?J{mtB!QC5mK^@U)E{*ll<J191pqw50uA0V~9F>'
            'r6jLuC?|r|32s!efzEX2Qr%uBJq}y69trYa>ZFLrl2jVr`da#O`?6EW`jN2`4%1|n(A~-fCrfvsY?a8QSz%b{CB+}uy_Px%n_;HH'
            'QWu~GG}$N+*NUsAQs+apS199ouDtl&$v6v)H^KSV9>40KypqLfbjO@H3>V4kF@<6+N7okYZCf};7X)A(H~G@Nd>f`ZzQi4bHPfiT'
            'DW*%zdQR8+TlXG5XD;=e>ndkfx6b#B-9(qV%bat~o4!+Guv)_3y>*1XEh;92w0PruY259@n+e_cFJh9vD4l#lHvt5HNPq&RK>cR0'
            'dunus-sFYS|9oPM;wN)ilA7C$p<iX($yv3u#=G>z<|d>=dW<$Y*H6>Td}}Ic`<!6B-D^X?MFo6vup+tZDo__d-J%v=?W%$(bq#4)'
            'v$zd(RjJd;m_O@t&-xYGgSWxS+v?@*EN%mo#tpv*I_2^XE=+Cg)1SY8^Zd<=r{iCLcz*iR)7Rr4Up;yH;`G)VJiv?L>3&(hf>s|`'
            '$~;X?QmW|g&U{g9A3CD;JZNsMRXe?-ul;S%x_R?2HFj$@Zt;}TaN2&4%Sim74*eMb3#o6oA^~>?RK-mW&D?5?HBQrht$Krmy1d3M'
            'Z-?~#0hb-aY_~6C!@3#nv5vEFlq*_L_H~moNVgi}3xd>I<J^q75QQF_YJ<TXcm~QN-7A8#=-|Wo^$EJr@0joY@OE>s1H^4+AyC$*'
            'Znad`#@KInhY>_v#+yha%R~VGJx>!cffGh;7B~R#lA&7UFd^{GJp+4rk;Tg)##Z5xQNj7}5as$T%0}0%7GYbtXJNB`lAvJBAmgP&'
            'Wn8pu%D3=x&sqm9MHzIed51<$yP>l{e+|w(pkx#{x@)yWuWoSoRrxt+c%~3$^v;#dyinajZeJT&&46qRa9+4}5f`c!=bRa6-uqzc'
            'udNwq5{<Rt@%t`#^wk58MZ*?2`PvVSKem81PAele*LzX2G3tlxyxe!d*am#<yyFGt_yC0D)oh=-gie^r5I0Jk;oZ0P2~H(?z9ZRG'
            '28lyKm{|sm9@tXYG)WlfmZmXX2U3OV=d_sXXS#+S=@Yb3L>S8lJ;!*Tqrp87rTPefC8<rs@dNlz=5>%mP!}p-V8jVWVwlP*@s$Ou'
            'FAfjI3A{Y9cx8R9LK}+w)D32@@qRbV#r6d2&x6(aZea4<n8L9aC}~7W0?W8eQiIL*UUMCquFeN^J1jGWih1svcih-aX>d@P#T9>T'
            '`luYNqBYX;qv7T7h_1`gFQVbmcc$0r!E{KMjit%t{mK2iRG1ePA2EZsL<J~W&@E#p3~`;jAJ13xWm8|?JvmxCU?cn#MN`>XdzKY%'
            'YiMtHQbW%z#v^;6&7$diC}sSyNl~2cu&Yf}I!a%%iEVmPiLooEFUPmbTeGvKd!C%cmQA+Z=I+~;6t+o&%MwOVp?atk(K2DT`iDZ9'
            'We(_lT5X~a2{O$!Dw>iqT16iQie%VXRYv00@{Ds*n^LG&njs3jQZ}Lzxu#6V^vZt8ZDNHIJISfqF}kg^A$xWxOJjO*LUp?}&JFT<'
            ';&pe(8BIMMwA;--hI<_;7koUCqeCJ+A*R>ci68E4tJTNcdT`2A2^wIUdkOVQQPMjv{cAL!P=x89o$oL7K(VlbFV*DmJxXq8Jn}`F'
            '3Ku*vL(N^9*SgwP532-{P$cL9VWZ5pukag4ScN*;@kJ4>7do6?zvdJh^s#4{RzmBH3hl$jojdf0+&y;+V+Y8OvF_JKVdWoXUZG>z'
            'B$(E;?GdD<cA`}~VEbe2_FCB)LeJ3bm1)9iM&^NS1m$f;))^e`wZi6*IUlt-{IChC8?~8s8;;`oMlIga3~p*OzAh9M@ziE%?UF9P'
            'KJrQdb{(4+A9;9tyS!#aM{OR~NfKc3nWGL~)tj`Xp75$lk8C_O`H9~miRFFKYh!|u<t!(CLsGa>8F98MSC}jlc<PPwE}Qz0%0Q7('
            'aNW@}4W+*g_9+*;?v8npaBJjDCKNT8@1M~f+Ou<9o#47PHQV6Y?eBJo_f_JYqx5}tnb+0%e!B@ylSVc{wdu{xNgGhd1Df|tI9B;j'
            'SuC3lYcP_MxYHeAu-Pf@xa*ctcN<#qXQL)4G_(TGMn+_nt9CPj#YT5jwb4l^1DA|e)&AXn<l^a(`<D>ViQVR~2eJhH*ao*)K7H=('
            'T)yTW3|@`#lx}W*j}RUzWwU)6h0c)96;DRoAm4Q8KUlZ~1+TWm63K)y>kAy%Jb02UV~i`E715u?*ZT0R;w74w5?~a_$<UDK==`K('
            'hz)GWVW4k0x&1eevPk)G?DTqofAwxH^&}mF0erG25QkvB@JKqmiY-y+){>4R&MIMtE-lfbZU1BC9_UPdIZvYtVuZg&NlNB=C&;a3'
            '{>@U6M3NL;X{H<L6KpPy^6H|6+w)+Jw24al(c@%U5uRiT`L`Aza(GNhve~9C+FG4!#BR!s8Md*0Cl6-$%s97EjE)DMG$kWW)oyw<'
            'eI-spag6uWo2+(gFlTB~Xwvv*4ERB_n&nq?;r$0-+Xi(tMMuV*DMzmp99{rb^Mz7@)YeEF@+@GZK&y(p|KeO8ZJ3c5e(r8yEX_&h'
            '7QqkXYA%7k7dc+Djx?Z39!lrE%UIKgBr79~wVTE$G+@%Dc*&{_(*yga-Zm|xR+==`vuF{4WcDGA6mIcV#wf1Je$_^Xr#zk2!Jgsn'
            'Xrt^3aB&{9l0rIrs8^^E|Bw_mwnl68t-!QHDv2IyE{zbOmJ=4&p!-KUOzXU&QX!y9hvuYD7vw5p%+pw3YO-SrRj3e)>bUB5x6eoe'
            'j>-pSo^uNI@eFZ*>1Rrg<U>(1U!EIh4G-Y+AS9iPO-8lE$zf`wU6U@uO=`A&znz4~#5KdyzXO2d`)|ZRM{xF{CC2IA#@Zb+<6OP!'
            'wMWs1gIA|(wZCApxIH(9(VhbPK36Uc@*A$uJi%PzvS&n_I{IYx`uC`qy+_k!zLvA*^v@BiKj%0)!tCUZ7+Uxhy5ALv8oI{8+)!fx'
            'NdR_YRvV-VP9i25Kuj+3RSg8<p`TpEnN6@hmS|bMvGi#&Z>pkT@>pC&Gw1|5?Vy^);9`xj!}bH24iI!kyWh_486+8;e>ijQ-F}YK'
            'C|_6Gb6-123j?o5sjTb$<^u4#4GnFVp3v*k?l^}Kp|jmZywwa>3v8$L(zsL5m^MZ7+7s8%K=|gNj7~3FR2Grgedq53?d6Mh50q!1'
            'nuegmwb)HI6c`Gr=9uuzoN4sIOR|igPDxW;gXu9%?TxpjzCC(}J|{5ww5Fpzs1^^`6!k*gp>;*XxQ$NMhNG?V+J;X(=(OkG0uJk3'
            '6l!075)>KzIP7#A=z?G?k;gT26J+M}DU{uvZRr>BS{U<pM_FZxZ067JH`L53y;qD(Cp<lmUEt*G-WnQH8GXL&0Be=p4O!ZGQF~k_'
            '7F=zKc8hcYY#bDZ9zTa&l_dV(q#U&?Nvgj~O(%o4b~ddrwF{04<m~{`a(EbuTc}V2zw3ipxl5ZGw458vS{(}Epyk2fR`TeE1J_T%'
            't|Qgb{wiipft$3xx>76Ez>NxE#mksS4{1D#OAjN$hX8ODwRt<M{n=8RKp9(&;w_7cm7(b`;5|v36R~xewyC9;NKWh6%H+WICc6w@'
            'YhV<T_c=HX7L~~~d6cY91*Iu(RnRJQ(~+`yZCn<uxonp?;Tx)ONl5X_$EP_NT*9g5S6bptncffXcnpgX?J;Cwl=%rA_53Q)t}*zm'
            'Q$|<J*KS8TL^$N@IzBlNIwdYdfUa*X22q)m8>9HpF*JLTnpInEjAjwZNM$hGsUGy!lSxF5?_*^np-3ADnM&NZw?I8ZGw9J+_}M%|'
            '5vkLzw?BpAr0<`xMZJoY`o+iE8;Tiqd9->po^0ez@8-*>QZH>l=*m-i%9GDD9(**Vt-2QT4;)0Lnz)^U(9Q#cI&<gp*qd{wF#fe('
            'U*^d)Mn8D@r8^gAD;Cj(?Y`BZJ$7MFE~AeNTI>#1>~5z!ig8!;;;f!7;dC6wl<8z<f9-C}IX;Q+xC9;&1N@pSrg!d{*Dkr=Q9s??'
            '0(Ok)PE+H-?r6)MQI_2;-75U@c0qXEjoc8O_X8~2K^k9T8WOx)FO3giA4q%wHlZpuVx41e^d-wwb>~IPM@Qog5Vv{~lau%=l8hPN'
            'Pr${119=bRdn)hHwSoh6LLVd6E>!nD$7OSHa}XNVg{`0NS=U*qy8w%rER-b%)hXkj_*zUTCr|_FYq3h+qjHI3_Bc%sM!>PPgs#1Y'
            '&7ri&=|<aEIOL8=c}SyIH~rZ$d}E~(=cp(oY&d>e4H?_WDFRtDMNYfxdXgsP0{fM1DoX6vGnH2qh9oZ->`NezB(c^l3^~ovyv|6+'
            'BpyF{`SR5p#0$W5Oe2Siz)?fWF)>k6!LDA$yf|5|wKf9KyC$zR@d#!=E*CHpI!2KV7?_2ZuRLt&vI$%d-PfePny(<z(<%%L7M4JT'
            '=p>#-z-~b=Ci!fG{(9mH;|id?grhh&ApD4*02Oko2n%ftW<j~uO3qds%>e<gDCOd15>e%MX^vI_jl^oC@&OPSaApqE*z3OwWu7ZJ'
            '9$xv=A08dR(WLSc<13yA&V~g)=@3QVm6{L-U38%Hk6u0z5#bdst8WU7`=IKzbZyiN0hHCSL#T__)VM>TPl@IoyJM;M2bk1MP>6;c'
            '3B%}9ENWRLX)0pOxkGuS7_X=3M=%pLt~bw~p01`d@5G_;zNNW)hZIe2cnXz^Eu#>$9Ip`t)rIXmd+=f!QSwQh`T1%QQPNG=dT5fE'
            '(G6N=nCq9~)mq-Loli)}Dmc_w8m{_>(lf3<0Jef}gD+`C&_F}Gjj}(-l?10+#%}54(vx@*;ZyODW0GNrc^VTKN*axOaEUjEH$lB$'
            'D)I_5j@dMEWMoF7S;4-cF1?DQ8Rex$%q<uB6?I6I>m&n8IQVRly#MWkBH2#@wf34*9m6KiV7`n*on{m)W4dEH{K7OH8ab}~fSc=$'
            'U8qFf6;-|7+2%G7J=G*<Ve0SqTfNm=_H%c*0o>8&j+Xu1w$ESlKHyR3SeinC<!9>!gEm(9MkwmNQ`48ks4LMBMlqKuOh{6|#+&8k'
            'hHf>YJVC2yhQW=#HZcEKE9zSsX5{5_{uQ^&TAB8I2A+0)CBvKD2pn(cf#T||JIY#-ZGOC!!P;(r2h=3RaPN0VtQ*1V9H;(CA?oHZ'
            'eru$<cZ^Qhy3yWWD|&tK8^YI(?#U>2y7Fk1lpJ0;Phw0gU%{bMpkaPQDU_QjihzaMg-ukyA9~Y?@i)^T9ho=cP5f8kNn>>5k7vA$'
            'CZ$RY-Q0gO>Em@UdIWn0g6VOx6T_VFYEIVy`Vu6z@;qXnjk)UUv!DfjcSb-D9tE>#nNXGp%3Ww)uK?JPcu)!0f_hVzs~XO`p~~=J'
            'bfF<F|NUb8{}KN7Hwu6IV-tuZrpXd8?;p``e`nEew+MW@Ip&Se9o8tKid@~bSXwb>7SGZYW`5;!N2Wc>FyYY(9d?vc4~Iw@2z<+c'
            'wC2)CAJ-Oo2fiyB?H|!upCLNyw}wr0giQPu!zF%0sDzC%f3*+^MrwPt_?@{_ZCcV&xQB+KJKUjGZ!6OKzb(ijG{^9t9C7hA`7{2e'
            'xQpuO^T%7LPhP3kJl8%)V8&U*Z~lu$W>{I8{x%UB)zLp9GyVrhX2_{O<mEVMuCvw=8wrPOh#ogSs!;Ly5U8<u{_^zEOY!REqdz?t'
            'PoDna`HLsdUt9OKk){cFy%Z$r59l4e7I_wJ0BbNg5XNbK_SlXa#`9N<^l&ZNWO8U-r*cc~6B$dUI6Za6Ouu~*3hUQ@R`wFxOoFU('
            'Bwjpv^YpbgI1IG2gmyYWx&-J}N?@4lY(ln2G$Bn&x+^78I($;z=OWFsi^kZ3IY8rdju+RGb%&Cbrl|y2q2)r_-_VLhc|Gx(6G{Rq'
            'jVz9l#4(Wcq<KU_WFraQD5IKF>(bYD6J;-9(@x83a${7EToA+MW4AuND-@^kU+{tIQg+`A4Y6u0H7R557G!G#Z-Hrs(Zr5D$7Hl!'
            'y_A}$2Ftm}KK%DaB>dM5Pck*~JNjT;cjUw}nZ(%rE@?0pb0at!g7lDxBk*x+%&jtuVT-ZOK1rMK4qG#HQ#{oTo$2}fs4y#fv`TTi'
            'U=U2JzcPT2iU0GLkDtDH`s35n=i>XPe|+)$Pp@9PR}S<zcBZByU<RB7CRxn0BI4mjn}Sow!FC4eI24^mY0@bI8sfDleP75n`kJ|D'
            'VQXuB)9^JWj30<s*qJ<=Vk{3xnRG^NfSN4xvQAm!-2(2Ug#Qf@NuG}Vtzvk59$Ct{)bpvcy6w8<La@JnY#5N8FZgd15Y$Y5+W;91'
            'yG)b+{Q`=*<AOd{EYRnR|M|F>A4f}ou9S}6V5d8~p*Vo5JG2Up`=_H~fV%FhkMex~E#B_k4(?q}?YD7J|NHSn{td3hALYmVKhh6*'
            'nxgxW<)@=vHDBfj`C7X&X*a4&x-Ri02Dpgwu6sb!yhcYkx(ZO6j#-vN@gx+>coS+JG6-1c9!dsXF&GSV4c1U|4c0VO@k}aQLh+SS'
            'QwO&IMK&vjB+C+xJEi|W>JI^e'
        ),
    },
    'video_analysis': {
        'sha256': 'e851221ad1eddac0b86c2a953e81ea601cfe4d476d9107b44130d64bd11ac067',
        'encoding': 'zlib+b85',
        'data': (
            'c-qaqf0Nrb(jff5J_S~(dPhtoSu=_A<e98;<#omz*E)8}c9Lgr-YE?wQ6pw3Qp+DRmVHxopX~m*y6<)U3m^cJlI$c`o2p%pNTShb'
            'G#cHF?nb}ge;rjtI8P#V7cZl16sBRaE8;>uQtPZ7ZL&?2mTDQ5(V~pAR3+I>yimnD%%i26?|PHd88w{ghBIi!t;X?om(Ep~F4bf*'
            '+htW*&7)aZmDwglShG!7lu^!~jeEV9SsBf<>{d-q)om1Q3l*1z%I;IOsgg1tEdWxy49iH(qja&}g!yfp-l(6StHC1MY{MLX713Rk'
            'he_|(bzE$tTy3&tl&B(p9~DDYETS}$!DKl+K-{t|A`>R+#fu-|i!6`w0var#n;rcd414pcROLES;i9bI%P_gg^0-`Y5XBmsLXAdF'
            '9{?BnsFray+D74RZyPTVW$qW@CD{VN@$icH2C=ZHj>zDcR#^_8ga|!}?;^F1@<<g~Z=RKFl?v4g1)vgw8odKNRS|9?wSeKPI0ZU~'
            'Fw&LUgmGHJ{}Al5+9vS==sE1g=`z|z@PBEU>_*ExUX}R8BHPAM0aNo9sEMSkfKO7Su!6Q(xP<A+)wY@^aj}jH7}l;_L&I;8ELocD'
            'R370+p$wEbrMHR!3R2)6Xb7$N-8PTz;^<y2l6Z^UhXcu@vdU9nv^X7YqD_|X6wH69iZZWYCh}<9dkTZb2^$zX1PXW)niktEEn?(s'
            'pda;xLjv;7n7qaKw*nZt%+xY2ZpXcTzu#Ns*+$J~s|tb7W-8unv%Cak0Xp$)(d)^xJkoy^)qI<0i>N5{lQP~!3~L*f>m;6QKtI7x'
            'et~?9pXoRH_mCRD&r$<^9To^e|3st>5^1&B?o?Q)bldB_dHdw;&u?bGJbCf+#TzwMA9@P@>qqZk9mF`|FzF9fe+l!KWLtbN+a}>6'
            'T4%{J%JCIX!7PvdRmHHr`$L8hZee_P_*?aXm#Wn&Uc}fOCj%d@LYPu|4IeHmV*D9S{SH7p0A@e`^wX=?Z!f-|y?OQX>!%m9U!H$|'
            '@oIMQUvDp7zIp!YrG@Qyv;7JIjW^jH|9i{--beEd{Sn^9{Nk%ShW_l;QyBK^i@yt`LBA-&yu^<-@p73&_!lrY{<EkG5I_B(*Xu2#'
            '6>>?I+(otU%ER;~8hj3-BCGO6G+S0V%c>$;41tlLJq+vyo=jiD95I19{zFc#0c>Jvq8Nu6Az42@2#LQ3Jf@SO2OA%reKiTx=sQ)y'
            '%8V{ousTaX82`CCXVRjqr{ThSTvwYg9j$Vh#w9G%Buj5bv^o`!4wnVU+boT7Z36r(U7{dBKo{@gg4U8=awK_h5u((_g&XJ6WHb*8'
            'SiWI?Q}L2hZ(&(K1=axhOTeDOn#y2}LT5%!6@ZkwzP71-eXX`h1u`g3E07rshbBvbWD1k}a98x0RN-a|T>}1=z~<S6e^ft#O-G$h'
            'G^9TE3<H5qqgz<6@bco-v$3Wa{Tr{GnFl=8fTyzG56;cR(F6hi5zz<koS~eY1m;scbMDvnd}0HfQqu&G;12$(Civ2T`0G?n&K>T_'
            '!?=jlFJV$e7kQrLgT9B%chE!uA2fUY4uL0|Fp2wh*DQv)`=RN$l{836Adj;oYvdY;t@`sgiPKHAi&6*Bx%y!L{7e2I_>}q1qPV+A'
            '4K-U9uRWT7RS_`Sa&}-AfI|ZHx%zrCo>1fH-6GnSYVdZqW%P&EPy&>~@Wr0N_IZI7u(cF1k{_i`{9ZpO!d+YduWIru+9v>{CbU0V'
            'cjBAfRqv~mw_KGi!<$%v)K1lHxJ$!Z<>11T5bv^Fs2s1_$aJVfcRb7RHogsEIAIR`O-`1u!spelx^)@Mngk%YKXt<vJc3rvn@*L;'
            '{pN}^Gnllsl1-x7%tuCz47LGiC}(@2;)c@%*$q)Z6;a*D;q?{?<Lx63vwi}@wohvI9`t7r;|T~`(ENE;53;K-u|BnzG`a!mtdNT~'
            ';d@IaX;Ki+uwdGR?*=gZ;T|Ff3_t{0o>l2`P&Y#`!BBk(|KnFR7k;TVh8f{Li&rz)Hx?1>d&>bRJvYGe1?|7MG4N;?>u7Ncs`6Cb'
            'M4&yEx%m)Ptp*rSPq3}wZfRaqP96+uRH}3)n&*sPE(eRva;OYK_0A8XAOr|;vQ*MU-LFv*UuP*>8Ms|8csD))V<O8yZG_Q-oB}r6'
            '0?&mvq5xc9gT1w>QZgQ3XWjrKswIpF7C4$D+jt9V{jWbfe{n%jb%%g2?A$+PNkUdloEIg*Lt6l~MSL2rz$Cm6^JPIQ!4e*WeQ*oN'
            '!VdB58SL#hD%zl77^PWtvj*ioyh$@qNEQWb`UDRQ46xA%jWK8|Xyt%C^7Gqgqi>)e(s{|g3KLKW(C$Fhgsm{d3>AhS|M3ff+Q7__'
            '$$>j=gw_<8lrWa?21ZC6e;e~~LiH<HQQ3V#{nB{AwsrIfqyx$TV=s(<0wSk_dWLcGigF32^3?w7`A--0Jj(ON^IBR&@5<@hyaKfZ'
            '4AKm#Iqg^FYV-|oR)ou}nD%A(ZJQ~x0|IatYeNe0@<SvwQZOjQX>75@0L>U2$7FmKG@8$IG>7dcBCD*?y=4F}e`t#|Q2<@7a31bm'
            'nhj;9?&xW@0hJkcn4kE8Swo94e~i;U^oAVRD#71C%e|mKQFXBeAVJvz#}5KjRBPd2Mu8sy%Yz(4ss-J`zmvjW2E9+CB0)s{9R&M('
            '3fxWKGoD1;l14GGf6SI~Zlp0kL3?-#S~E&YG=>|d3CfE&-BzV&DED!>2K*J<q`c73fsmqi17tfoBX%fRL^eHvM|#CyUpM*&X>|`~'
            '28)xT+QOI7a;#oKxA{G8O4xrI-A5vtzycaYn{BzHkHO>y(}EELVduez(E@mw<swR=MTv0Vy3lg;pTGy`9l{w$J8Gc@8!`M5Cq1@+'
            'okZHQLRq+l;gcK6+QR@H9G03#t1M@Y67ATA{n8-Bxy3U=t|`sPpz*0=gP?-1Ta3YAMtlcB+xGqp-n`7pXSkDUZT}y(@;R@P3Wf@D'
            'AGCc9^P%r_XS#yTxf}%W&kTKe#~3|+Z&}gNjYtIjiup+E5OLm07kF+qYO)^9=SURz$EL}V^tc7nG|MZOE;V38ne__o2Q2`#%U&%H'
            'fe+BtMSlQ$+BgBf5mJt9g74}nG;fgY=oLA7-i?#&9+eJ+c+ij20>qs+urKMGE1P)rfGmaCEH}`l(cSY~M%Dp78XUwU2o4h8VV+gr'
            'CNvpJ%g8mA>RB19@6jN+g@p<Wwt63?pf`j$PL@Xcp>M#;qzRLuvRMyzsBWX(6ja;!G9+@HtAR%~k`1#IKBX5Z8uX07AbX#wHz)|B'
            'O@>>~=swGrY8_QMOBT@ZQJ#T*cN3#;Iki0O+PRDy<+yk8<EytoxY@6-UVm?S<BB!tUPylW13t&U=TVu5yZSG7E8;$)|7Bkw=x*g&'
            '{xH5oQ0>TL7~ibPLkFfgkOu!M^NK$w2Q&T=&NJE;VPv{_lHHLzZw=(ro<}e@$@Pd-q3+NvM1RebEMrvi@CJcEFY6rkU=1!_hbi}7'
            'WHin!sW$vC=>GIPybBBb<0gKekmC_xFCzLF%@qx)jMhnfONiWt)Kl~u=m~4`kCq`07JZepae|_jhl_GX?gn6n866P}Rx8wQXY)8c'
            'M-Mj`A>q9xhKJ0EGuRE~H#Mt&@ADuk<NP^S;v2NRBj%yjm&hmS*)y=6Jgq=czBI6gHP|m4XN$ruJ6m}p01;AVDwK^iN4xPoEbt%3'
            'XJ3Ey=*g3(<d2elP-ui_2d7`({_tLHz)n%WKKX^U?h+V)K4*m=q#k^p7K=P49qbVwx|r=yy+$ceC@_O?J&^<;)dHGTDe5@Oo~X5`'
            '&VpXSlc9va#(FDtr$BEF?o*aM?&Rn%-KRkd%<(qN@$hVOivpW}6k4xG&+9C^rJrrxd_?1(aV&$n>ue-79|G%+XrYhRfa7c)rYSG1'
            'QIg#xc#c4fj@0be?Dv^^jtbm}&d|uM3Mvoq6yOqr5W>kqf3xgQuqy-h=KJgGQBho9TfpwK3XE1ZzIgH{lk>?$Ra<oO4iH@QE<_bR'
            '5T~v>`${WBa_nP)LxD)9r<(6dun&JlTWv%Z6CNyrY(<q-o0=d-!Il(Fx2R9oEP%#AMJG3~AU0dJVrlGTS!ZGfM~jGzT#)*!93{B{'
            '(}z*_aXIA!v!ntf^a|~65<p+vMCrDqKS!~A>#nb0Xu&Jx6K7r8A8f)c!kV1lolgj;c@UmY{#)Pv^ln9>5R}nqN(*KX-#<DFK0iB`'
            'Uu*j1v{{_fxlIz!-@)>x4uAj!OIuXhS7A}|<M`cdwV}Uv)!kQTldT7cWM~*J=<Dz<i<dL@3xV;$1lfeQ(d>PcXFQPXuBIAxBj-Vu'
            '^C<V%?9ft6z(~%>xum6`SaR}rK}rkCMe^6F*~=<Pev;RsNQtT6?xK-JH<XTyBI%i&>Mt$+R$GFB0;x5}cpfX*y9#JqC0chRcr4N`'
            'M_*20*<2YVlI6x9z4U9&^vR1{ls4GRY08?x>DPcL&ooW=Dg7->uw86s#p%~{q=&tD;?$nReKTMN|A$QhI=69yS;V(t9v_?mQUmmk'
            'ML`j$05Dp-w^Q5#>CB`IpcY)fhKW>PMtAX|e!SRL{jR+q<03#H&dS}kZUDP5x`cR?vp8L4=U^RW3A}q20z-EJb9Dow6TCIUC{#TH'
            '-d$G|;W4`BU<}xXsm}tWHY}MLdaPg`qZ>LK<T<#f;D#t%tn~|Aa34;LR6H4~lL&Ms`4eH_zv11@3^)^ToS*OjPSR`!y+(`mjGw9#'
            'RF8|3b9L#!tHIF+@DF|;;-N^nJ9)TzaII<Df_Pvv8JIrAm!Oj$aqf7Tarnv9c{8ZzR!`=X{~0<HHno2+bu_JJXwVH6&K>%|5#G3F'
            'r-38r>N$Z`i2n5|youx}dLD%vLS*_F)XQ**wje-ejln>{7obbPh6V46>4!dSv(O6cAN!DPQ}oZ(<H_V<sE~{^vl|v3I6N4RsY!-Q'
            '6Gom@1u?^v`Ok61T8)E0N;+#YqU^ge8l>U3)TML1{@Xl^wg9nVkHalS)C}aeI9y?nxCD(o8{LfY8OR9GT*1f|JZv1uedB%S3}cp~'
            '*V!M64Z?Er;>@_;>o#ON;?XvCMk@uK-j`RNa|f`&mt#JF5uc`#_G3J`Z}oG^#Dg+cMa~O@ktW!OM!UXQtiWM>GQ%feH6na$(`+eA'
            'yR99(>S~4Wp$UB^eGu7OKD9(eTPN0Q3IP~TpwF*7Ko1S>*+Y;{V&JrO(V~M6*rc5dedIRy+GjlPD@*FOfVSSYjJog2=a%;F2W=pT'
            'O=YHtX7&3A>m<A0=zOQ2vjs5(<<VP<%sy?P1AUf<;D**)3I@SBhJVa*hM&#g9@A#l#ykxcV|sLS>wp*%f3@@3LvQb>MJ3^|#Tmsg'
            '76aS&po1oBwF4G2v}RX+0;c6<OXSg&eE?agpLTB$^7jNwg*PEa{HVH{vDM9qbZ~;>ISH5H7WMrT-ix&!5mL|SX&6{e31Rktk>H4^'
            'V(jeSG(zr7Z{CF3k6{W_g{>72=Ge4qZ8SDPq`LVN!SH)|F~ra&9Dshn3W07nTO=rhKNvcm``@F@BQW8m5B^~c95K+)46i%j8-jR$'
            '{7n(ADROlg-2#LP)K?AdFZqKS%*Eq*t_0<w()b11s$g?uYEf@c8v~?*$j*h7oNbuc@V!m@LCYuOaJigmYzFoq2W=3?EqrW$MocsH'
            '+5kOg_3J^P=WC^A68}agZfF(+dtx+XiSeq{4H&pU2^!~hEoX-fP+9<JknB4}fU@pgNAu;t^rtrk^8iJ@s2UtOLA!Qp5$8(ef!07p'
            'hm6A8!m(1yY&!#nz;mgoI-LyB0W(_y0921G$0EKRKi=b^TmvBBt{ZfLGqlI2XCzvd@uD<R<ORhXTUG|BUj_MoDB_Pig@MEqkc6Oy'
            'A5KUY1;szzj?-nB=iv^lfqc_=L?dJaf&A`l72t=U)sDyGs|Pt8qLYdFPVvbdtQj3ky+r3xnyK<WQv{8ok<l0e0{hf4GD&0znDKd5'
            'gglXePl0Pg9*!~IdeBhli+oBJv*^Z~!fLUKLJE?Gr-R)T;MG)~)_(?<6IKL?NOp77_(3txNR=M|p+XLig?0mg8F2SS-UXP)fmFb?'
            '8^(#JrN+y5$aVzy25CxAFVD|Mr&sk#p!O6hJ+)R<aS5%jobxQOjtAkq7}O0$3iS8QWna%_|0=-PM}Ha^8Tb;hvC;KZU;C9&&T8O#'
            '2RHq5C{aHr%g9!4;0dEYR9^=V&LEJs4hk4d#>=eN9zo$?MO3s7m2gz&O)A)uf=7TT(!Vvr8~mGi7=GEk=yMDVI>Q64JKRt-vG@nf'
            '6;5kG3rjwLbwXw>(ST7PZ|gd~S(~}#PsbS6h5Y3{guVi^)lst%f>3`|r_t#2tfAcj3g~Dch7}}HRz7ydFK-wW#`O(q$bmVr+2T13'
            'Cl1W!IF(@@N5!CdtRl*X{=N55MrC(Vo&YC-sXH^q8f#N1_pS~!XMtU#BTZTN8k%STW1}C36p&dqLQoT<v~bUeV5YtvPYxj_YCCM('
            'JT$BIWc=mh1I<^kl;`2%whO?2Jnjf}AJTSFYn>M~Z=fbpUe6L^KbF8RI;ey)SwL0aY&hE`CHtIoy||Qsk~cIIK5&xh;&T9j1xi+&'
            'E|LoN06tgXs1G)eH{p(tvo_F_z)@BVjtyhnaL)==8ihHg8d2hqOeP3xj>n=C2N(iELLdY{OHhO{R=;8gCHc+B`=J5WpkVyyJ2Z4v'
            'Kv(>>DxxJT)Rf?3LgDDpI9U>)s3=DolA1^B5ZyI76KPSEr4AiLl~13vfH++wSwZ<&>#WqLfnlmL6KHdjWlPM)QIag?_}J{AU>%;x'
            'qM*UkA`tW$_sj%_>UJnf#HO@PR=S!at+}xa(T1b#3kOQi;(Kc0MrU}#ki~v!S@W<rjr7~b^09PA&sEhXLYxp>en#ek)mvbD$}G3J'
            'bPYrF@Q2aY;jrEyw08l9@Wr1!1l?EH0Av-<>;U0Qe=a|FCQ9~0-LhtZOPej~R!E#HnekfqQgk<kr+lD^uBr9;pveX`H|fRYuoIEU'
            'EIXr{*S)br^g*9Ci4)bq=F+tHfmxpcdCi-@PF?D0O)Y_)5a;IiLH}-QQ~0VC@tqL9Wk3PDd7A?zFRu=JZy>u97HAtqz)1ux!y}j`'
            'GT%gGJFGQo!TzsWjp^7!Erl{n;+r&D8a{0R>v^FYC14ZX;CTAxTZb!l{gxa8zZ=7V{$LpIww*T!_MqZ7==bXEjK)-qoHh0d#Nwsd'
            'gj^fcC><>HQX1mXRFb9d$T}P4FODOVk3|QOK2zUED;)0~#=PDE>xHEM5x$RD*f44&DNzL=^?|if+I3m45ooNDjMej!L-ILOm$Zh`'
            's3f8`;2y1Y3$2du9^Bv<poa5o=xw$^uZ4ubX4aFTv%v;#X#Ldkxc0Xh*wN?>?eq9YLjKV*&N-txOh!bY(cP(ieC*=n7-YWmx1HU6'
            '8?p9Qgul~9_5o)JIc=Rc`y2b+TF?E4&f+6?wEe*P(&j(-&i=_?Jv7r`yeY1eV4P{h$9m$#HX;_BS$WliIuxtjsL_TdS(y86m@Ye^'
            ')3ot&WzA#9Q`K#M@Dg(EjV~oTD91nu*QYhvV+s8(GwPEKsKchyUW7#zn-Nj%c4vFq#p#R;hZ&3(PsQ6EfL}fyPwK-0*G_wiv0BF!'
            'Lol@$)F8%s1(cx75mtQcPhds_hM|n^&Q6eN*I;ni5wKR>27MG^*Mbm#G1Y?z#_DOdna3&U1|pZ2IHl!I7<dGxQ2mHJbg=BATrW5$'
            'K-YX%FBIHmRyZCE0zl!OMDJ*1OV}K8{MrtlypJ$^77WhG=yWn+Bo)RX6iG}0X3WMCN_mq<(K3d8gYyy!j+|qcAFc}N13JcR2TK{('
            'V-pD=P<0x3^9;r=7!qSg0~p`!(A`mxDIST?8``1v&9^)-j8q8=+=**)ai9b!{weP6@Da=q!TLUf8G8BZt-3+#W33Rkt^^k7h$8cL'
            'CT}Sew-^Mj6-t~mlTp{MOD!X`kykrLS+mEj&lgXi`qZLD&gs6ue<r84Xfe($^jJClkRQ+rMQTL?eQlQjR2wJ8#ZO*NP|clI=dQjI'
            'Ewrla8q0uZje5}HRjT!#Hbc&_BIStNj!|gYEc|!_cy8I4>N{&pxK!x1Wk;*O>H!%*S5B6k7lCofhuv2;R;?q^sPO0Jl-CKa!13nM'
            'cm)(ge{|lzd@}m);pqMU{-2}&^S@56zUU7L1rx*XMrPn?Zj)m8>e7icxN@@>_T%07R!4f^89Uk3e2hNCV2{>V5BkXNziE8CROTa!'
            'O<3Gw8Z{p*fc;lHVATJoADhVveBS264D^-P;I0}a6g1<)s?%&dQr2Pc>DVfw;H>D6jjyo0M@Pn2)Q+NNe1&0qbiCtvy!*awBH_!n'
            '$1O1tpj}pUrc81KXV>pv6b#vloG;ZQhq!)2QfcaTMYXYH#b41USkEc38$N0AU>xX&=bZP@zGxA62DQcX3(UBQ1~s1y)qhTcz!RK!'
            'm}I;I{(SL%?2SV&3^yCbUNVM4$Zr;L)oJ$x&+5J19>^vNxjC<TM&FVG9hi$^Ff)n)z$8qHs1$74@x$@}DBhikKBFxQ*N782)6>DD'
            'yv5!|J69^XvorL$V%Xp>+c@z^KkTB_ux+q*h_97H0yZQdd2WuRF7GelH;eRpoH%FYU$K7;zQ9-RI3Fwxt8i^ok0n75pA4G|f^_AU'
            '1vVL<KHftH`JkI|;PqtKKL0R@k8Q(_&NI_x^loTQDN<OH)rORwfkO++%i`-)hN`wqtBFn&zNCJxkhwq=9jdc~q^%|VW#8`+HsNUl'
            'jYcbaOrJ7hv~+pGc-B8}z^J)FFq7{C#JG<1aLgZADJ#%4bNtYvQj9iEA?5JAa|Uc-sVdUgagdK>X|7!7zxmWY5U?C?YjiCa+8^Wf'
            'gNK!!0PU|@`tZ+8o#C+@(VY+RnDsv`a*3d2vu}W>%t7M}cEE?vwtn`797tPK;c6;94h1ou1UFouuK!2Lt>kKVSlLOZv`@Z;VbGEw'
            'ONUdStSK#IIbn<es%^cNLt0@9g9X?i>#T^<0Jgi@kLi5uYEyI)IWGn-)eO9#wh{CzhTklo@SX9T?lZlEhT_Knk$e`BtxQ!*@2g4T'
            '#vOQWBF8zTJ*?Wd1;p$;IBy0AYdUybY(1~nRd?(0Hro!`wI|YVn+~TYu!e;*>~iKzJDxXTHXS<C=A&k_Hh0dn?V;P$oqCIeXpocS'
            'L@mH#0<=a?yBam@cT4ulsjU44sk?_@)l#V)k`7Dmg5x_4EI?kdvoBP`wRr$&6Sq!~;uP%y=&uL@=|Ak8D{^~Kla{Dp)<&iu>INtN'
            'rgw66d1C7<Csz-kK(v4&4n7FVQFOXt5mz4u=BP)V&_qi*2+q@{0A7F3g0PE#iV;|gVLuwnStW;pv1HS;#VdQsJ3LN}oNqhk+JfTx'
            '&ovCJud*+a#>v=XxBF}rN6FGeaSI>!J5s$vN_0~HaW>gHjx=&M*1;R>o3yb~n+gl2N<0Xqb6K82inGHyM4TmdqFdsyPt*VdzV#!@'
            'FgyuN<*9lM*Vo3OdB2V^L<e3aaf&&_PQjPBP+7`RIWQApm6TG}RK_mFhPjeVq>Ec}<rP(>i#7W8fMnTf1vXY8<z>SqmRFc?g-brn'
            '&I(Fv!hj&kWybS0iY~$un7Xbql5eb@RY@|Es+8u{8Vk9yV-{m)lE`gEdC74}g^wf`IS=oxIyp)NSYOu{=7YI;ktLWFZOYYrjzSCw'
            'V-E;Jfph`RM)jDQ=(@n;z)){!dJT>ZG`IQiL;b^xGM5f?0ii%Dwy+ciMUO-6P@SJ*{f-*WgS9IR{a0GH|8a@FhUs;VzxAl`&9Q0y'
            'Wmbks%QkJHJtiN|U^3*7hrp_FIqly9r}T%qNZEANpFH|NA3VS+h~FpCMpyqrjc`eV!K`A-NVIyDG7r;JYliJ5tD_f$43B{3b?ey&'
            '{`c|(b&r#)^U2xrfvWu0K{5P6--b!sp5t9KU0AVU+)(%c4IFu5nB>Z?t=4d?{@!$FFLS@qc`$5bM0<JqZmzyR#KwnqBL3*?9OSZO'
            'NAJnC2YHu=oNVtw^D^tZ(&io1Jp8YEcG0lTjCViNx+B|q+NkQ($r=ru>cfo0w^@83JdIbnYMsu{KBfF@tFprr+_fv*QD~{*5!VWf'
            'cCA5Fn&TvjeS}YVKZ7M?WHzTVhfb*oEOgEeOjw+}@Qbl!GFuCqm*)Pp{Q*CEXj|aD-e>2|E>mC$v9iJ#%`uEjmonyk1i{n3qRy<e'
            '^g{1AurVPx*yHCB*Y1$wi&!N(qk{6FD%`KJkh91Vs@9BU8ab?0R9!yXB!2<|)KD<C)^y7?-eA*RuBrrS4hU{g0__UWM$m|jLR3GX'
            'RZ3Xd4LPKxjB_mVk!;4jXFtDqF?;j$$xEq5Jp2CV*H6f~`{v>)9^rq3p<@8%4bo?m(cXSTyEt~hRWVr=xDTmJq*lndet)$@uZ&Q0'
            'iKmaSXCOSTNo~l3U(fMVp--0LoJfcd(uByw2iCkJ-=^7p+A@G2e}3`y`A;vNU%UqN&#d=9KKU=J3i_$@>eY`IFW<hI{dDnKKEVTd'
            'MDthjg9(|{O_I%Fusg*UD$u<UY&!^QTsRN5h#S_COeHd$Lxgu<r|)yfb*&4*)_|8eFs;VwO#~{*3_(RwH&^dB*Y>FY{^Hf&p1hb{'
            'yu5h*_kYfQy?FljAKp?#!dG53B`PU!WF;l9a?+u)M;F`$k?7YSUcI;wo#aGvkx&z~sucb7^hY|2L#Y63N_5wsaCx0l9JE^b3p0(P'
            'CDt#c(`k>1dC_o3yk6@_a+wuc6V9Tfye)EFfu3IF1@=o6Xsv}L=h}8XBwY*XUZTr&4Ruyhm0TLaSOJZraO0|t$Jaml<Pt`0+xj0@'
            '!Njkv^Z&ep$)8<@=8q|VB1Q5FEcyL0)lZP^XSR0&5u{Rtc7h@0p!tb}lV)Pz(=0^q?du=kXnBQO#I<RDt-?i~6?jF7URu9C`DMs<'
            '9L6kTkzu`3zXvmNPFgUmu%(VD0Sif|#(_1SNx5Q7^c&#{69IsN;uhtBR$Uowip_v!3ERC|SLrgx<&O8UL^mPUp_033wjv{NX|qTO'
            'Wvxde&T|~WxJI1F2?L><JR+>C!syU2FfWn+V8d2!P66Fcs*pmT`7%K!%5-H=#VLE9<;Pix_u*YsNA+_2tEOvkuqeFww7Ud61p+v6'
            'f3@SEW;5=5O)?7+98)FbMtWpFpZBTKIg2Q}m^R7-%19LHbrY4TergAG#7h!i^;b9%^wg<~TSm@DH0<0ho{Q(Z1Pzwe*}>|i+7brF'
            '8qGlogI#4PRVY*SfdPH4zInixZqZai=9p_CZlVRAA--2xo?>qs5byDO`|1bcqZwy{%3;?j)}1K_)XC=!ey~m}u$XwYp~Kn9!79_}'
            '$uRW8Rmpu0Yy=v?DJTG;;rS8urFr#8q!cDE&8yWaA}O!)65$05?DXqk47LwEz(aZ(-j`>1K=tL>e|-IoRbAhdX%|NNPMtOpk!=$1'
            '5?}<zR0j3YBhwB~PW9i-Qv-vw{B?QG5U+qn#TxYB!RT~Y!{VTV5VlD$)*E3|HU>>otDSDNHno$~L@4vTK9bo-Rv>4++ofK*y0wXX'
            'DE^h#5fJpdf^A+?C<p=%km>M?sWS*Ci?!Z{bvmn+fGPpUcK)fgZ*#t@^|{8pvxVVz6VOqw0P8b5yu}L*;vDADDV~N}ZGni|;pRtx'
            ')LFMcHrX==vS*&tZa7qj(oyTGT|WX4AS~^gvSX662v3c}P__n&Z5--aCae%vDVryHtEeNoSzke<IBbk(BN)0*k>r8xUM%cVLI}qC'
            'iB2&3#t(v&xi@a2M!=aLx#h&2xiMN!IcF?~Hb7|OFx?G;pu0<_MuX;Qj9H^-wZLdQ<f)z+gGnr&!k@tQo2QP;+Z7|Kr6E>!+h+$K'
            '2bkOvI)whKu2AluVo*Hz@-{Z3#jS9Ug*0p5!qC{xlr-CjJr-5$8CatQC8(fTqd^v1(%bOl8x$qc?6|OJvN0Bm$%kc;S*sH}&5IzF'
            'v5HxVOvV*sNh=bEGanh4oLPYN`hv?&D}7xH7oTV2AC!!1o@OK#=doN5B?*W70c1CLKz~cO9p#vpOZJ<qc74RHM!MF|r(O|Mh8tbw'
            'y2Ju+c;yP+614w%Ae$jPt-yK{I3IiW{K(GHWZ9sx0<1z#E~G=Fn`g;QqzBQeeg00_1-7sV;#%t1b{Q2S*)*S70_{6B>9wV|EUk#g'
            'a?9m5P{pVNa70wqI9SEglZ=9AkkIyw4DG4%c?EP5kHAz)>eP-x+J<S%B}R|9!=5WIT1w1OtQeYG94=3-g7zzp5Ox0ibLw{Q^*%n{'
            'L9Z#k_SKi8j{4_<-xun1cuWntwu^(B?OJsJYhvmcljgYg0wy(pRhRY)1pB@;!mPR~jqy6snHFpP&Qf$bJB+C(`R$acAWPm~jugXX'
            '1za~0C;aQU*cQ-tM0vw^vJ>o!o?w0sk<PtB0-6gcrTEK1(xXm389zR6s<XJz+Z3>&RZ6L$cuPx53N$ruKPKJbH-T+MnN!?}%>vxV'
            'Lu2IMO}h!()cA$KQ<-Q-&<KWp3JT4yi*9FDoHc->rlA9hHSG4epjGJ$g@-E%Yo;JLJD}olnBk!SLxWLwjG?I=GR13h2ff3wZ?Got'
            '!r*9W$8eIDNS?aiwgt*HA^^E5h2|0&+JpZzuIIhI<vSU~;RqTXCpn$Es$>fzS~Ik_mQf>uRhK<8Fukmf10udN>!aCSOESkEJI{-B'
            'H6^JIK(tuA39aTfDj?;Lq($nx;rd=tNH1bs7s9GT0c8HfHEpcW=E2Rt{pklSIhIbHklSTRybjY_|A?u1zV!U%o9EwOG|!hLV5pIu'
            'e2t=K1p8N=KEMZ+CakPZ2c@%Jv?QGUOwP*qer+9>6RQ6_<DxjB`d1Z4=1h?559ycs@CFT)lcITkNRk%kx9);&)TaDXx#FD8LpdE@'
            'N&)c|3C6})g?H$Xrl=^gJ_P)lRw|duo8w7sQF6F<if4XOy(^FP#Z@%JY)H{v3SdKFRTv_^nHCs?K<6w}K14D{RQgDhC-3>2+PPp^'
            'ZG3P{w%ceVb74Cska9hJ<(oRikGRC&f)eiN&Oy2}4886IQRfV25+#+=Jq&b)Y>u2tGz!P+*O)F**Pc4hK&%Z}8J|>5Qeh&>JhK@e'
            'oa&Pd`t(}PtV!#Kb+nc<Yal%$6Q>xr9eKf4+VkXYNp*aVx4?Q`zP9##Q3paagJ&D>>ezf<_vG$Dtf*QVVa0!uq!6S*-pj*U)zM{z'
            '$YKX82{lIoMyU{>1@I4YdvPaDK^H9VHnqNM`OG8J(CAyKdLM5+nKU%m2Isw29Zh?k1Z6NXz2b6qNJ-1QHzSu?8u%$J63(yZfZuX*'
            'N_Sj~-7At-#$XEB$p|Pj$8%gC!r#qetY5k)9Wk(m1~&lv08re2z#o{C1)n4hcwZa<WgvAAL-!$^@)2_*wZoQCPRAUgE&3eSp~JWo'
            '^Atz1x8x?lBehk*<l~jjW-xuNd-?*Qk<;RR*r}E71>`EHdfdNDJvG-b9#l~Em!zViZ-kVSpi@CbWVy6KTjZpM))qEtSWV|);2*XE'
            'Yep`2Y*r};wg~ZbaM|Gymz<C3IU4)(L-iUz)#of1V9Z*sD@j67A4Ydyj$mB#tbZh+w#!Y-%RSeb98)N>vxfQ6<zfD0M>}Q{iNA$$'
            '`5(L+O*=e1Ua-Fj!IgObW>k|CGw}@cSu2RYutFp*UcP<)`og}ZYe=;ku&6ObpKJ9R`IEU$bLKVO#zmD<G70>PZ>p>+bSmoL?5jzE'
            'F$xllPYz%0QMckLNFR*pc1S6Eg{mx!X$#hf!C7n7Y8Bnf^<Q;}lmzbCdha$vx&qh`3w8RTgO(;LvGAf>&lw;Y1aqpr24B4+@h7@Y'
            'xB{uLL>=;6-}VCfJYVZuSb_YgbZfPd0(rp-h_TQnjF6OP%~rbpXr|R(jF>6pnr^<aSk5~{<(4QgQMd=9LzC%eoqY^xT3Ah^k}(uZ'
            'GL)ltl!02N7tS%xJ?w33yhA(z^5>$PWw&FFYdfRMy{W;)e?5Qm_W8@dBlMBXIbMlPsTrWUQ~rnq?V;C)Kxv9`JO?J**Vo5L%AVA='
            '=X5-~NRX}!1h9pRN;Fsv=Eyu?Kh=gLMpu?PjuF^Gj0W%A)0_<;;zMIg=xcX3VH&T{b^`-WZa;E(682jT0OsINWA)i}UEgh``WuY+'
            'R?G#;4&&nD*OolE_Nwj}^U%&^`mM-PT5%MuNSQWxD%G+`nd%gz3(M=;WIO^bQMC50J*R&y(L^#(sxL&QIEK|MF!I1NV*Y@@HEZ|M'
            'p@Y_=#y7`C3;vrPtwkmj%3LNmb8h*yj%`>qagNsJ)$%4ndgFVGXK)B;IT<WlV}<!-<!spA_vdO)0CJ$~-hKv)?^FL1@n&bX6c`;p'
            '5GGvkb*gVGLr;n|M53nFndG6nnd<Oh9gu4C*dvGK$M@k~Uk6Z<A7E;gNw)$;D51C+KZ1|_&Dx0>IYkGx?=}Q(d(hhgg=@lIvaxMR'
            '=dOvpVb4u<&>Oy`9Qb(|^^m(mG(BkCST3^|*a4zLYfTK*t2guq1?0ik=T5GLM4Q};KL$-#C&U=z2S5Zx`5o=2W=s7Rck|z>!oHIe'
            'r4oj5MCzuBMVg|o4YU%x3|5v0o`IRz0t_Yq*$FxZW@hLey1p~i$om4Nbdl9r8{Twb!Wl-7!2pBtY=$>B&t?yF_xQrfN)=Yef^>sb'
            'pA!1Qd6*`#hzqp~w-QQ=nf1v{gsKd1J_z>kCdMnL7qAuvAMEZQbe7yICZC>7VG3rzxN{Emq`P$Svib!u;bML6gr)xYH+I+ocIbTs'
            'Mdm(b1;Nc`SsH3eLd*gdz|L~U(&UZ1Z%gqQX<PvT(%Zo%E-(Yq72Xz)2_t#eGk_fF_1pvK&bS+&-CN^srrxt*Z;uHuZFx3y4m_rg'
            'yl&6sqq{X2$3Un3(CoLFI94K%-R=P|=GiF;e~1=wvNXXCzh6?HIZ++7KYNl?@aTIV9v#x3CHuS{R(+iCE`(*k0hV-ip#OosCd-wg'
            'jo0=qTK1=cJI_(kM8`{`)HzkQGeBt3<_kFHHbHko6nwPg>#}-R{lC%F>+CR>H)8B7qOG$f<z4^`KGe}Xm)2#OR~RPp^(1&0gA}@r'
            'Z^OU}7N#l*dULJ$1M@-U=LZdu&YD`2LESC-BcPZ)n{AzvN`K>I$uNR$DRl5c_U3+S#yYexK478SqQkG%BM%*w;A+n<t!o+9_ov$R'
            'F=ui!Xyc06X04OfJ7=d&zmB=8)b-S~lwz^3Fm3bPI9Jb#7}z^b3=y^NAZ!Ltbch55-S{?MwF-#NODnr2ibrk^4m0-w2FF73gwI~6'
            '&UJro564Io;|jyA(Y+(h;%Dt9o4Q`J7BJq802ptA(s1V9C{|qJg-%ywL~-Y2_Fa0OWMV|&+x_>B1<eR`9llpXxfFf<YUn(qWAonL'
            'YCkOzivGe3j#}x;3-vYy3MKYWpJ25esF3o~sI1*g5U3abo|@V%MO&K#Xq8K3ZBF+kk~XVM)jsjYF{SSYH)wI^<PrbS9`vIFI5e94'
            'yQ4z58_v-(u=ZAd97nrQZnS;y&LKp78B<B9<-n=k;A~LiYKuxKMtjkW*;PAvsi1jTE3fu@4Rs$UI<*AXVO~gE=PKbLUt$Z^gG3TO'
            'G}P@+G3i*@e3fEyxv}L$L*m;BG7PZDvwIcZ^EF9!=5<LvG3{A8ifWTqT1|$nWY~Lyl|_*K*4%?1X=>YatbhB;ic!F#O+;CeG^WcF'
            '>GFh9B+YX2%{Wu;m#sD`#nP&ZwC<9-_U%o89W=g<F~S^U%(Iq6kJjgLH1J?w@q>>thCG9)Rrlp7kqM}cB8hOU4Uc^(uU?tgrkf9h'
            'u3E>7d7Rr7(a|J}C4Uc%dQbO8mAegBa&s-?;199i$lu-KXcG@Qy4jupnn@_9FIcoHLB3T*oQKIr`P<wwj_u)MhTy~63~%#7nyXE1'
            'q~*M;;2J-Ep`5xHhPqASd8{f>IzewbegFq6b~TLxyJk-BxT8As(GEQO*(#o61{oYz=!*wnm-?ET@=QI+krF@F6qqqimbQK|(=Tf#'
            'cgg_Sa#T+atTn6^gJ(Ug>yJJ(jfV$o!Pdp(r~b_vQ@T(?*Jho5fAQ?e&oACeAjcca`{K}j+NkcJwF)B{r|$WwTh4FF_aWgk*Pp}1'
            'Wjz;=6RT<am56PpUf8MOW>P1bobAPK!aOdk1%2%^wEv}v?X+r1jPURjC9iuo4p%$#H9#j=b?+#-thkvrX!K>XE_FSN=-+O>*JqY3'
            'L*u~UeZ+a-KLdw@F%T9n^&qXWlMG%H_oS<Cst>!o&+XU?I*D*K)P9T8@Hm?G`FM7QXhI9W20fi@){(oQ)z40Se~+c5*gjzIBR+P}'
            '+7Nnm=!#hux5h{c<a)Jc@>BD?Zqk?+VfBXl)4%q%HQURfisK<QD!BY^d{@lm{85<7=nBmb@mRrPEh+eZ>RkQnswq?j7v3;6e8n<o'
            '>7>S2K-kl5x}MD`oR)V)LELPF4pa_XowYA@K9LzwP_B0smTri8C?YlREBJe6<(^1rxwx7_SZQc<5H4zbzXdLzuCq2}N(f0{iB7~)'
            '&rq%9Xf>`{L4n0qtgWvbxv|BqWs2O8VTu%|%0@n@5BtLfAHH?siK!41D!YJ`UBwctoxU&W>)=5Op=b%M<Y;5O4K>^oZ`=yFEBv^v'
            '8(Ol!D&rcG%~vL}qKiRvaof-NiWLB;D@9VBPivkimV>M{iN?2ruf1Umu(FY7?+4#<V**p3Md`2_&>kYJL=>({%1r0B;|AYZRV@iy'
            '6UgD9?P>`emyb+^2;{(kk?#)gQy`2~Jq4;a$@$>z7Hxh=+A-Q*bhkA=M6aoSez>LMC+vV?e`8xgZMa$ijHX*)p|Zl6lq$tvz)t$D'
            '`t`}{m(O4R{oKCW3Ij!=&9>aBL7LU~XO%4R`gZuE2H)%T773m<Il&<&bnRs}%C;(-(`j&Sf^*F6UKseLaGDE>ssYon%&NsYT2jFR'
            'Q@@@qi6Tl?6r@{M*6UjZan*P{HZ9mX<rZ3_S(+++pKBwve*`G<5-=qBmf{jSzHTvAu&wvC<mMFX@a*x|Gdij#X8?W$u$g|#_=ffW'
            'by%#yHX8HigP^XcL&bDhneMEh!v*E5^cJ_y80*z@jPto&#bt0hIs1wt(9gbty?);BdyWZ4M!R$fo|Die@7K{g{%fEsD%fU=#Bs{F'
            'ZQXK)7Zq~KkxK;NgEy1`LR3&-&PhF|b+rjoQ>-{z#8}fE0|Z#$N*Et-W0jQSI&A~XWnl<7hG31C)n;3?+E9cKy2mm8XBHNVIG)nU'
            '<&c8M>6#2a60yRVaf>UnyqFI9L$uM)F(qB1C#M+v%gXVXu^9BLay9y<?~u`(YQZ;Xp(`7OVt$%jvO%wHAd<`FgsSVNsDzNdITSol'
            'jKb^rjsuqE>2Hljg=MUXolN_biQ4TCO{i3y8}s~}igw7UkJo2da$_%?wxx_1@T>%oG9!#M40Y|3gre=PVGk+eRk+aC#bcC(euR4%'
            'IpHbcDih@el{m{WRMZx^zSK2(2aO$~nr(<`g<<{9Z4%N|%UOcSVpM-K32~c5;b0|gaU=}K)^0)K-ysu&l!Dzjr7Rb8hQTPcw`19Q'
            'AA13^a}#pmRCMRufr>!Nu+0(tQDgPh5p*_?y|tnnF^xq;@pHOf71&D-um{8JG+DXShqZVPhYGUCxw1_ckdj9Gg8BH^WA^&um**G1'
            'UVK0M+dqNyg~~8K=l!=P-3+9~CdSBNAn`3MTCj*??QU|suV%3z!%N^^+X8X}tKROmO&5IA^WH@N&Bcq0r*8poJ&q?Y|Dm?o`?$ml'
            'y5n4-7qdYBq^^v?F{}W#XB|la9W}xf9^fW31m~I#?jexR5101z_OBicdpI<aI^$l?dj{#L`E5g<esKDDu)ML~Htd8=bcYw0;fG0>'
            'Bgb)akB3oUg>(guOtbET=Xz=IvB?}E$9=2xLr%TC(l@(Hou!!t^pe`!fXqv({nkmmD@|!1=&^O%LBU_*O`;pz01aJyXb*K^CwEFM'
            'O!qM>1uR8o3l4W_jFK^Qy62VG_tK<HNpLgfO9zFkUQ6k6lO`m$99o5@-SX|cciQnm<_(z6GZM}ATYX=>?h3`bQy`vHvoTmhq^PwN'
            'S9jpHG({DBfd9-)IW(_h3&Ifv0arfp^|rIjiX94I5PDSCvqr)`t!fa;RYV)rx}$`O`ncQp(JS0Tm13*W<DW*zvU$B_nnmuR{+Im0'
            'U-tcH;d^yTYF_+4PUA<${KM0O4Lnh>5b#PiG%@g|h}>TO9R=xRucG_fz*IW7ix2d*vOu9R=;ZPBjJ|sXIM`g|7v#-+*?g-lvU8=G'
            'S#~5mN}_v7)A|HV-9`59g@J-Lz?LluJ!kfs>i)dx*4*FTHs+n`C(Z(Gd=Ruls7sBsw%4!y*3G+)d;rZax?psn;J<H-fHFcGh|^R5'
            'TQk}2$MSgZt)p%W>s8~d!Ip{V&C>e~Nk?wtX3ARM6%YW*ebDu_kzsZV^RUYs-LkimgO;B=pBgohK45jQ?SrtaKfqtOSECw?9<?!S'
            'f)v>E9>zVc$^}aH3~Mc+jBY?PzPDOY<wu^Ts>?1~0J<$-0QV7QanJ!%#7GE4BfLDQ*i<-@rI8IsPEpt}9CS33^GF=phX)4>G`Z?t'
            'KeY3Mo&fm5wUoW`Cv<yezUlfftjY}LCSF*g_MF3r8!V-3ZyLL&-99!yZNqq<(Y)V=tqY?7Ggx&|Bn6DuI9}4U#hs;~qU{<3v^^z|'
            'gJO>puHXrVU$}~DzlApFF0X8pC@wJj5gknf18~Srltv}Y%f^PK(x^x=MW@K#Cl%(6-NZ@%${z62xzo(wAcy|0I5a!ouwzr&vkMK~'
            '6+U#=^Kt8H=<%j~9EqAZ-;G2HE|1TtE#cgpRP%@%*+es8O64|*iu%nmQYRnyBBK)(-4^j}t!p>TYMvf`>?-8wypS!$p?iRZsjVW$'
            'E>2C9+~$~^Q&P=pyHsSeQ+H~q&!*1PiG<TyHBe$+>*`tyEb7x}PzMHVttoD=9Vtt{H=JP<=U%Ti_XVDK@9f+5U4YiD3r<>0XiOK='
            'k;9@QUNvs^JJ-@N;?x}Vy4ATo#QbLTwNHs=Z*5}vxDH!d9@%G`gg1SXpzmh0W>aoqdi~S=VuFzbkvbm*dsaCn6|?vgx8jT(+|!TJ'
            '3_8}&9tlZns=e<{A~)Q8NL#R}e21Dgxx{L*(q;dA$JCK-yIRDw?M}fle!T8(oMJ+)tHbTj_({7wF^8OM$MhloF}r?*Y*D-HwjUg`'
            '4_S_(rp#<p-ve=u?ZK3N0t8n3i=q@WCsB`zFgP>vy+*U5+H68f7ur9WzD)H22<kNBxE!2O9!uRZ;oeGe3=oKlBr5C;w!@p{;?&~R'
            '`gHFUv^?mGYVkI8OjV1Esc#p(>ZG%0OcKSr9ZsKw3kw|Ba3ZV(n&u%(DA23>$9S-g9kFpxxo38=Wb9+P)>zHXYhv5NHKkfm9nE)K'
            'ogY*C*STCk0yT9>$HQ@$hkJ2DGfhv8=**+hZiHwuw(ePA^Ca+ZNN+g^+*@H|tlqFz2zmwI`9>g4jGDv-!3p0L81r{ERwFrhWA!rH'
            'O2K%kQdnjqt!zHx*wB*0%fjW(7;&AuntN7nY?L+^*0AZd8n!s&h7I?cx3IXbjCRA*LZyCNuWqr-Olmh&13WL5t14PD7}}2t_25xP'
            'PCr;Z^NpjJ@`}FU+H%5^t|ClWl{;4-sPn(%5AH2Gh@sIWn%9<pr~G$zN!Npp_T%vu@Va7%KDM*9-!_aJcbR{;r~Km|iLbuat{q+b'
            'VyUdav&&h=E2fqnd@Uy3l%6I2J#4g5L7Oqv3*b-vZ+cE%AIT0EW>Ye?!)^Jl5k+q~`XwIuT=kP|5hf!lQ^c{nqr0<qN4_T~_WJV)'
            's#-@?j%xD4D*k6l5iED~EoTl{rmMg&)QcBCq87tpF8YXsV+FR=9JY_OjY8WN(m!wAS*%u0ta$yy8JZO>UjMLR@ZN#|_^&oEW8-|m'
            'W&`b9_4tPlNu5x@E_b!hk?8#O9(4uT8-5Wa+87QHX+G{mrj4Bec-vz?^4{4}p^f{X?Q1WHyYb9FRxlBz6~?i3#Fe(5?>utl)Y0*q'
            '-?r2)XiDNop4=wza%Z~>y=5j^IvDUx^R(5|CT#9^2N0JYT|GK~(V*g0TNhGY+T83OeCz7zsM>N|vPJ>TFbT`mXXYTv@cKZkqf<oD'
            '?bz)nXg@Nst>5!zE|W2Pni^G0bZFxv3PYt+fyS$#BLdXR|1q-GEoY8}lfInLwb-1lo)#GOijU78<70|9b)KI}g~xuh$!IdXm#*AH'
            'uM>Pg0ta4WR|zaM5XG})OwLbfIRUT_R^d*!=GsS=>}*wp`SOfzrMF8>#+!n_;@qn4f}O=COp-m7H2<^`ogZ7BvuVBHZN_^E(=cA}'
            'B2gm9fwGz1Aj4(2En)tx;zIh041J749>b<>@dNB<Bf5chi1BwfNi>pZ&QZOibD203aB3<j4pOe>$R!YI8e#uV7i+MLG3z5Z*X2xf'
            'tw><sn<|$(%ZK*;-YOOYifJ#Gbr(Zje?ebwOa&N*N)kLQm(S2-Nfx#<f{Q^7rAm8Uu<(E-r6VZWNa9$@l+~BXWVXwyvZ6vJ?3%?q'
            'Qc64=$J<>xr<m`;t&5Ll&+DIG&VG6R{l%-<lb26k{PWH8H{Ytsskb)Kuoi{Z2F6!>-$(g~>s{iM8JvJPa}1{(H1cVNZf%V~l8g$h'
            'h-J-PJ=ga9VO7m72DDSasE%ldsaaDZ1_mJ~H(4H+YwyD4&(yW%+-usQKo#5~qL^wUk@6hfEg7`IBT2TFo6byM0*<OZ=?Y*tmGfvF'
            '-br{W-n&+a;`ps*v;wQAa)DT^<ce3}VujJHSC>)y`8gGI1rv<#C*T--#22Jr%7yW{@G5MCRQ9#crQfg|0Olx<|A-Qke`bwIOR;QJ'
            '#u8-*mq%UoXkSr6y8#tAYEOiw2yr<&{Rd~G-(pUUT*&+pS<jzP3aNRq`axGutES`@PHU|Q<PMxV!y&&xe!B%rORdBA6mi?~wd_!o'
            'E`CJDG6#5Bl>0r^mX5EoWSX>BTRNh;lIMBfQ)S6M#r{k^NhL|g%G`<h>g!clh6&gjShyozjA&Utf+aSRTplTDO6znTYWZPb5kJy-'
            'c_2neDvnTVWz$$T3X{AuIX%HBnD_=wSb?Q${E~2lCZO(utl@xl##$;3Q)v_RobDRo3qnkRUjb!I_d#2x(^U3sy3(aCd#Lz3%Uh>~'
            'B#bv=krqXq&4!X5D80eVC#J|93Rl>g!X&$?B2?C?;4j8lkN4HIBVUD6tmXJbt!<8{Qzt+F^U3SyVzA%RoMPbmGE0}08r)`S1zH~v'
            't=Pt;qQerbN&*|g4|U~=;x@|@E?|4&E$WjZTt?{Z!J<w;ncr{?egV>bonqc(vBAm_>5j^9$2%Ak7qwi3x8F)Z255aN#BD-nD*pc`'
            'SjPy+w?q1=O;n~FsJ=r$<CE6a&RZ`D2aQZzA{Oj!ua1|99R;&&skgUc)=}{Ph)P;c0Tri&kn@8x)M$#1s3|GU9%N35c+0BaL32|^'
            '<c-Y`R*WgJQXU^k`j4&&_NleN_SXPw>`uY(_|jgzy85b-a7Nz)wv4u-8-pSN<XeCtQ#?AI6h`Ffn*kAo0HZV|V2`Jwa%q(*dWW^~'
            '3oF~46BhfZt6|D9<_E1a+DMF67MF#&+KLJ@kzu1f3ZM&AlxkheB~nJzE6g#vM?Z0>&L&#s8+(OwwW)g8V-k-Wya9<qDmZhSGObv{'
            'Lp2RXEm({{p%AvQ!$EK269w}2fK~WdF1|r$Tc^Rs$-oUB_|87UtZh!=4K7~NZ1t$4gpK8>Zyj9Y{`lHv#b~)^M<WX-yzXbv3U;JU'
            '_uV1nbb{2i+_lZY1y=drXSf28LHSCBWgf5aRs}YiLM-sL9fx$$s}kY110#lRQ=r`oojsvbPFS#4D}c0iZngntT!AJ`YoL(Z3kp#<'
            '*vD-F@@|qg%#Ex8%2`Dz<@N1j<}wB1>yG*6t6m@Lb#D>gx*gXirQM=_X|?ENTWNi27ssJHNxu<+;#ODMQz-D$3j}t$iFsAi|J$yZ'
            'sm(c7O`kUJKCGV^7Tg9q11~|#o$*iG$zp4(`tIYlv(A>1UHv1<aq)%@+yFv3E<)I}b|LCDpp|2`5#^Kw{eRa=RR4}usQwKLP90{^'
            'Kco1R^}4GB)j@-Jy5|h$-=GQ=-glATMCFL|jgj7Ju`td4t+$JxNP+6F;&;(fE|D1-v4Bv6k1B7)GPt>&9jbF>x3*+{OEc#{({5%7'
            '{+4EcMpdjoxe7^Vt>aIwNODx2<A1a2Bu+L+OSG?7ccMFsBcyle?bHPaq*f{e(kU7PMlqIN16EYtpi`J`w+Pdn?XF<A#6X5vdo2QX'
            'Q2=Ky3ty37mprP%WCrBX3JeCiGb_&)5xY@Bph1F3O{AtJ-I*k4z1~JUjyICH2q%rhK5cY^3$d9*s1g<T=1Cx?1?)~c^oxU;L0J*!'
            '>f{4Jc{p*ygwcjKSx8R*9my|dAHpf3;7_~qE-uhXXSCmm)xillT}Y##FjOaa$JNPo9Ejd}DQ1c2%7LXrb$s*|B@7Ntj-Yd_X%)ey'
            '(~0G~_R9?QV4Edq8#ym7%I9sB*PTol{=B_>tUt{bF)u#=vJY3ygcvGN>ZHZEft3FjL|E++'
        ),
    },
    'approval': {
        'sha256': '8e6a6e6d92d0b51fe6ce55d06c96ec76680530d63e2335bf2d6961293a8f8357',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;NTXWmSl78o}m;*lutfF{ocPpt=bm>$n5}k0PNM1=!)~j-nATT5mf&dp6%d|@WeZTIW0R|u`Ih)=1!IA(>qo*%l-v*xNT?A>C'
            '#$k}CZB~@0OP%UGDB~<ut7;RZr@?lcXP<F<8I*e1>;3lAM|J<x<WBu^bNS(Ftj7O2xx2sXogVzxJGZMhRZ*(BR)M-3zdye{zaOij'
            ')LT`qf>N#U5}g+6{N4Td7MHZD3Z3I}p@Oiif+X3gLhH0gw+x3mQbAtE3p_AX?}MU<)1{gfVIFVGsW7dhv-`Z#GZm+0#$*>&8uH%g'
            'wLMFz;Cr%EZFG>z1xsGV1$YfMTfEY<lsyoYWx+-(h%f2!R>E8}Z*i$kPqE=emV;8d6KSe2tC9#C`>Yinq{oKpp4Q<VX;$J%Wd+_N'
            'fe(lf?gc8Z=5xq|8G#z^WJ8X+bXw*+C=r`2h&d~Z=8@rcHyfx~Sr+sg-WBy`xlSXUPpdLc<Z7goc+O&h(1S44kjeE;%@w4wtTuQT'
            'Eyuo7m8E!4f7W?VZ@?@levb9yG}r%K#kr1V%Fl8&TLf`JfBap|;<Tt1i#UuyWf}!#fUCV(1QJQMCDPMv5`=n{CE#q<hj#O9lZo0g'
            'HM1I+)n=o=gh^GzpNZp@&h?;2yfI&oD=lgw4Uzb&?KX*x#_xVQKmFJ5zE|0T|I}Yd*<qHJv^{J^cA_JQ`-jQ(<>dNDb^iYS?ai;}'
            'SL*&oUEEyXkFW2Kn7-Km7;@P)!pn7-<x$aWVrpJDEmi?ofqICOC|+u~i{Z<%(VOqcR()frrT$VvoIw@EnOZ=)_~UaNY54<Un}sVK'
            'uBZ0pet!m1r;sTmRfx5k7cC&*mdT$KDy;IH<X{*as3OC1YfSL9z<A7{2da6MrgZWC{%)Y4Z@R>mMHh*(a28Z$w!!j{T{=#|RvgtH'
            'xTT$DRne>E1yd=>O(2N^n@UTp3WGe~5m{&m-t`#dq@e4pT+v0+zpA!L7DUBBZL4__7b{&bQ?wOqj=U+=<ztqwfhzguI0UYFp4VeB'
            'Os9*gta3e_s(7=_ax!#yCOc5k>zQkSrWO3e{`q^6rLvrMQ^tS<bI1M-#p3_3S*qo^ZBP;c``CN@+3OkW`r&Fk{TT*#r_Pk`K@_28'
            'cq6|^y@B#X@buf~4BW$za{5Cg?)7`UHvkoqr(LONrQpcr_+oN5xw)=&JrLjl@I`S@Umd}90=mq&1%hIQ1rcEr)nQzaOU7lPlZEm('
            'S)|EGsyxKcO-7DfusLVs0qHWq<*X{VRS9aqL9!{~m=v>!mwmP|(5rorq|7f2)^KCmrkc@C9z6Sc@6&7ua^hx-SOZ)Ih8btW72d05'
            'Cv0Hx7U-6O`wS+>At`|~2$bWQd6I=|;5H}%vc;glask^lI2>t_J*^^;VL~Lo#+F?SANHWoi}BqZuw(lE>il|Y#XtRd{1NJHHbP2U'
            '!hJ+bi(Eroc{T^`h04K3H4h5j17I;r;&3-2&IpH7!eB%kV&0$&89jcBDh_&6ia0w`w}KCrc-W}eVB;1+R8!~}_O8djj&G-qInpna'
            '%keFl<de5V2=ShY<8!ORO|TQhr5JBt&*)1E1x!wE$MDOd1QQ!lOf2?j9wjA8t{~7Na51mq#Gn&4Sfw>#1A(N0z?IIa6FES!_73&P'
            '_K*Y>Gh42*Lc8?h6lPgSo`5$wQNH#Pkb7Lk;Y!W1gLNlEAcM_06A?>HDJftmvUv}%tswQCUtcOpO7JdrDvrMJuy_F=e(<c}c%L3T'
            '5RGDn9q-dy?DLli+mSJI_8<tbQ161I0D|3)|KkIMd^x=xUqe$<gs02NJxq;Zq^I!DD|NWyWN&J$mCS<m9(e<Kq?19}%e5$9xI^xW'
            'y_ASM#nfU{4{&sIJNa>PeSS4P|8P0E`I8I^v`pazTFlyE2=D4edVzE#NGkZ^)CibTDZhcXC=ScMI{hdAG7_?$H#Un3&SrL`vRVZc'
            'X^gRofeK*+9$I{>5iyX6Je^eOMwcfe^*|O-d>W{eMvJ)oG#n0}aZ$3NY3tqt?tOY5GQo4CbDM3sQ-&a5swge=-dtaOq&%=>;1Z(@'
            'Tt9SN&@lJ|qNBlPm<-I)f)$y5q_?z|f-52>yT<|37BMMxYLgeBTf_g%+&on}06RAMQ;Q^6);_=q79fJ`-%xw)pTCJEfSvtQ?0jHw'
            'u5r=+0mk;^J0sQ(`67$Cp&$Wj*cjhXjE~ooTsU6TR*H=lRl<Snl(Q{tncV&xcZW=B#qtK`lsUNLfI}!Mc^@T<Mu^r(`6nC2h>hX_'
            'i24cs^YBSD-EdA$RaZz$UHYDE&g)ZHczW*3#b$xM!F1w<LrR`BZrHcA3aN2a@058Oaz-BcFZyv`IJL@}O4L15ZDwmEbh&C2O3$|l'
            '$SMaj-E}tbPbo{;7mVZ|9l^BLP<oXj@g$c4sD#1R*qycXX^S<#7RzhUS3d}Wceq4$kmd0*PO(MK@~1^LFXJ-Ng;y`r@CkJs`uREG'
            'ln{*L5<ues*CUsc<}%BImbKK*k(Y@{?qe*nW%ezDU<#u5J>NW{%Dv}|#K9iFw4mZ3NJH)KJvm_4=r{blQ5T%WOQxA`j4KPK3eqAP'
            'P;OFEL}k`3(gbdn;>c*9!Knc&?^~tv0twkGX}4MLYH)95`!yt_m(}x<igJ!;>ZxAf@sr-jNT#{&Xh*`T-(2ZC&+RL7w7&)C`2z;('
            '3E=q$Zn<x2yX~m$RPx-j$r7aIo>C$0(|;pG)tvJJ{H)W0cU6)oAITPK+>rdgUpEd``ioPm2eb^jL-G9K{@v+66xZuR3))7grg9w0'
            '%YA+^;g09^hdK?xr0-Sb;`AS0zdytagudTz^Bmf)-IPjCKb!x)$G6knpf1h)cg=PE#rgHk_2lCGO8xsCsz81vg&n^@9g1MH*1Ljh'
            'gkp<Sy6Bq>qG=NrnG{P1d8nRa!24HQ<Z~~qt_*|ZG1wJqN9k%&-)Cw@Xt`dZdIz2r#3kOWDB$HJ4x}{@9<e}?H={WD7#Gw(<We@*'
            'RN?DL5QPixxLBLAj|@71M5X+rBtDH=1sdcOC&?*aaB4L-#EL2+U=s;=7*(5X(V#k{Ilv<Y^$CI^jN>ztoRAQwBtq^y;%~hT5Hzy9'
            'IP<*$MUjz*Iv1J@kT;w`Ut*Gbx_`lB3~Mh~2Q`3v>M;BnV<sZ7m$ng`2hO@Y3BQjq(##BK!Y`+e#2i5t018Dw&21zLS5>+uaDt+9'
            'KM6MTC>WV1hg7Qi-+%YrzkLsB&EXqu@DetL)t2<>Go_Zq%>8`<*>)S3D&WFjBmZT;AE*QMfh#(j<R9o9+J1%UmT*w{vW36FfXd9Y'
            'PH59it6WjxYx8Snz-+G5>fx9K@^#BbGNqSWEd$$4F*Hr7T>mpP-6o>6aLn^f&D?t4>AG{n@!jNVEVg0Rnfjyqo&}{Zrkr=yNZAy`'
            ')NuF}WtSf3IeWcn*EZz`lCGG0EJlpMJZS{J)1~tDa4EhUkN^?PPdSA)I|PKRgQX@jLfTdVLP{>oC7a<E4;B3q;oBI1AWGV?LA2YQ'
            'YfLhyjX3-87v4NZ(Anr6_Jh5^-PaIyl~<?%iRRz(XXVfJI)PYIT<g+)?%OokGC}EJw-wlMP=jtKjee>9)OWL)PD*9z4U8{P=7A^d'
            '<+eBOZGSJ(=yE<JP*rhA#iu&4R8PpjmzCOO>$fO_f*qa=!eFiDG0=RUXj~*30cQ<_&0_^D2C8o~GN@tu;k}L87fPbZi9-L+ZppBb'
            'Mu9pZiYLJS(C|aML8yect*L(pm*ATST6!5LQJf!T*x=dWsBgD%z*0>2rP-8e>#)CfOD*S~w<jy-<{uW3^k|T(36Yl8U521=<R6IV'
            'p*AiIw_QV>{kMno-Zc#XRvxdv$pG5&<jl`u<bWWX279x6DaI(o?N_b!t$j&4%Zxiw*WYh0_lI#IvHLY^TtKuie&oM7Wn5XYgH@?{'
            'l~l;$gD3`~G-QD_-Z2UA+Z+x|6M^8`Ztz>02#oU6W?8#bbI7?HZ!RC)_<>EE7;|vH4Yg*!2lnc`)T9K`(w}Qe2c{BZUdaktT|=7|'
            ')I$ayU8h<{JcX5nwoU>DxE3Z=WHV-Ni7_djb>PxY+EhARWxxuAdnY=nYpzH)Rawu>YUR{HD_vyNq;D!B+6)RI`r*_`cHxvdi4|2>'
            '9Z-<V{dTeE%qp?s8nD(X+H8QvJxPe0o1D9l8+BLl4MNdYsrT^7)@qzG<vXW8fgL_{nV}xo!Q`&dV827bVV73dT}l0ghu%(F67Y<<'
            '%YoeyErzN1cx2Db=*gErMMfAc!E=w9=Xhj4$0sqPm$$?6e}UsRTmw*oWfJTF&g$f;#n6dm=%oLAqPAHA9jyZ-gs9$Ph$ac@SiK3p'
            'cJSKskIeP^4n+6&1Y{<eZUg7O9wJkVn+rJV<sz&3IKJe9!lc=L+cKzAjw-a!GQtiVlBPJ*%A2T19$V0X49_%Pd8mdrtjQQY$AP(Q'
            'M+)*b$35NvkNDUC`!Atf(`u%)*|w25fkk<n=w`@R560rcq1Ozkn#L1NGN8Q|2OO=)Lw6ob|K?MtN%XqibZn>Nru01bWLhujA6tFU'
            '^4Ln=QHxaFT%Uhb<Le*CS636N`q?B?sGCojK<n$QK>dm=YjhwjSa?q-19dX|J&RLe$P7<D^`D38Rze4nfzhU%I#j5gkgf$WrRJMM'
            'nFbQJ6q&_mt#%diyZAI%qi(3#P$f)$10Ex4(}EOS8hWMN|8PpzLIsXKM#EF9yUvtqpqx--4}x?KjmX_2BZvB+$$V$1z%q5gztjBL'
            '+mC2++(y$W6~oJnf{+A+vlC14gbz5u<ar(yOZ_-I!sAStx!^0uOl;-eo~s3b!QYrk<<oi|*V=xxM8l{*JWx+k7b@Ku(J!hZNCX5-'
            '@3A?U;||aj5=c35z(YS(R3Rn$iz=Zs-mpe+#38#r4+VBm`$yF5*-6JqHv0&`({e6r<Q_$0n4ptjOfShY8XlP(!a$kiTFRFh-!|u?'
            'UCjZZKB*;}mpn@^hw7Ykln6a=zN|FOTk2C3x&EEAakGPWc!Z4W>_jl+gef1M)HAO^B!16lcbpj`Q}YY`(i{TX`S$UC&*@vR298Yl'
            'on8fR4+^F;4^^4R%T*~8yFjw5_Wb`3P5%GNo<KWqc769aUC((;ea-SJO+ZnAZXIvHBm_hNg$i{SQQ!_YI*EtvcOAo)-nW%|-jVf|'
            'p`p4-Lvq|ubIrc0LPW+mjmaC}ZZVYIO&yk5Q8J#Ub-xYuG94+S)IUd9P&*_Xj6wq`0<lRv5b8Bp*e91q)v`*flm!vD%WhNoyipfs'
            'jQhj*-3=U}4$(YllQe^Ds2E5iTw7q_c}vkr>fgw~#GR)xO4TRT5)A+-99TGUT7wmgvZAbH_<Dl$f0-A2LuJpQ8lHYRzMNcLPDZx<'
            'CuT<_J|~FlB(9@n9nPbebap6!y8+)WgLAZkIb!*NKyAXi1a@w`#La0hM0r;G;b?xe!EhkH+eB9){Gp|H(#43sBD~Nd@j6{@^jeXf'
            '@3Oq$`Ne)N>G#!|ww&&z4l?_eNXgaBHxxT$e7OWfadEcdCfTzlb;NOpG|Srh{b7<l>Kv^7?X1Otz5cBLvDh~m0Rxt;nsXbBecg0('
            '=p`qy@}G_{_%QmH?_mtXVZWn>g=fJPMtJw({QBpK1&191vH#?5!=XYAm108*8mfu#ARI>VAvh_rGzS3b6rn7&y(gGcmWXw{Jp^{J'
            'z#N`nF^^3V6zpdA)ZhPquO0ue*8WVL*b$*yHXf@JNuD`;dJU0vipHLs-@+q!{M^Pmd-uV%GpPP(V*J5p=$K?dCrk0@!jpkc{gz-s'
            '=O#^hhIsqr1mEW_JzhI)j`X$LGBp<?Zaob4?|>$?WtzVZJrmGa-kvfZrrD!!4;l}vGNjI026RC7?YpfUE&{NSe^QdwY04ZqpN<u-'
            '_qZ(@Q`*!yoSC<^kDwndZVvU;>{0c-lA%>9$ae_Y2BkbfU8&5c9&*>PW)4yXWs~u<xVp#p>e>c{Bu|Iw7qN$vXX`Ze$0b}UxA7ml'
            'FsDYW#rV#YshL6dyR>9n%(wY>p;dEh`W+CLNfz2=(@WG0m!M_^XXfInAcU&l@s<dG+rhQwe14(s?$58rtOU33!Kc~C3XWGAb&1gw'
            'Va<^sI-_`gb$3Ij3~WvlQOYGQ)|{|b%Z$$~%PgPghcpj#XPhp#Nnre!tF6;hY50BSsbf|xSI)Nz^0fv2-V{0^GZe)d;kBy*IR?mQ'
            'qaC+4X9DDio%6Q=m=TuOhwTnGUn%N36rJuOz3;(mK~-H-Fs{ZjTtZ<cb$bV_<?2ZvMMvN3b}fZ`ujjt1=AzXrtzBzxPS63r_PpR2'
            'ymU$uq5a~vJ7vgzlLFC29eJJ90Wxl!SJGnYgeXZIw3?gD6cK(v6%-19_$y-J4naSlVybs^*5nGa)yK9z{Edz4*W1zut)i%cbRBoH'
            's+BH+t%P14o`B5ri&j8q0)<^SA+#DgkPN=2DU(_JIKDtSGWoBOWssB*!M`QFA`>n)drA7j^KI^so;_JE>Rb*u!J9s_sa})i4b?}J'
            'v6<`>Wv3zaNA&nc{pw&um0Cmvr`I?4s`<Xnw2%b%Ms3{V)`}s=*jYEVn{)%P+8j1^&h|Rb)2Z6Ox<GtQyG@PQC|FQcG^!<2E}-GS'
            'EghZKD)F>_sY;|wz1t-|UNjyX2)}}Q0^J?R+uJ;6vdZ-|0~s-8_zV|G+&Tql-^EGN!D6tRD!thJZSLAO?+bza)dujb`AV^Lz5r9d'
            'o!?*lG`^J6OTy|v{WiJ(>E^?|^JPR_)(uiV1%*1Nu$>ZaY8av>g_ch^#d8SrlMHrUp8~S3BK5ERW=PE58IGt4*5q_vkh-;Ys55KH'
            'b7l@t(MPXxV8;2(mA~R5$K`&Dnn@IyQOKIL;~0Mj5Y*Emd4+Ts`I{n1IsV=>CMl6|KV0Hr)ZRC;NBiAxZ4K|he%$^nRQkL1hiKeq'
            'uTwd3Is=0VeWJw&CHH!otrp<R?MCOZrrl_3-MZ(v3%2Fap5tq+42xdf5cqPhQq?oSzo45jxxN@*jei-dpU*#DPFfYR)63uu`cE_+'
            'KEhf5g+7J>6w%?)z6A{uo!-S>g<RXo!3x>leehcD(k)i)WBbLbV-DWPwV^}L{O^j@BceN4jJ{UgKAV6mE>3-athYmrPQ$gvYV-ZI'
            'pfw#lo&rXuT-%zHuI)m$thR}M5ZtE!Kh@BBk)<d$kQsz}#FWU{`5<=P^=(cwQ}h=eWGd`8pX_$(YBGerOtzxzF*OOs7Glvn+LGA_'
            'm&WpqgN#^cTI77$Ur2^gA0^p8&g%{81LJ4*r1GKjz4d0y@oCzrFAuc_p+`5=+#t#Jd;bGEZ&}L'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
render_utils = _local_pipeline_modules['render_utils']
video_analysis = _local_pipeline_modules['video_analysis']
approval = _local_pipeline_modules['approval']



In [ ]:
ffmpeg_bin, ffprobe_bin = render_utils.resolve_ffmpeg_binaries()
print('FFmpeg:', ffmpeg_bin)
print('System ffprobe:', ffprobe_bin or 'bulunamadi (probe_media() ffmpeg fallback kullanacak)')


def find_first_existing(candidates):
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate).resolve()
    return None


## 1. Girdi dosyalarını yükle: `hot_moments.json`, `selected_script.json`, `selected_voiceover.json`

In [ ]:
def load_json_artifact(filename, override_path=''):
    candidates = [
        override_path,
        str(RESULTS_DIR / filename),
        f'results/{filename}',
        f'../results/{filename}',
        f'/content/results/{filename}',
    ]
    path = find_first_existing(candidates)
    if path is None:
        raise FileNotFoundError(
            f'{filename} bulunamadi. Aranan yollar: ' + ', '.join(c for c in candidates if c)
        )
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f), path

hot_moments_data, hot_moments_path = load_json_artifact('hot_moments.json')
selected_script, selected_script_path = load_json_artifact('selected_script.json')
selected_voiceover, selected_voiceover_path = load_json_artifact('selected_voiceover.json')

# Kanonik sema dogrulamasi (Notebook 01/02 ve master notebook ile ayni fonksiyon).
schema_errors = video_analysis.validate_hot_moments_schema(hot_moments_data)
if schema_errors:
    raise ValueError(f'hot_moments.json semasi gecersiz ({hot_moments_path}):\n- ' + '\n- '.join(schema_errors))

print('hot_moments.json      :', hot_moments_path, '| status:', hot_moments_data.get('status'))
print('selected_script.json  :', selected_script_path, '| status:', selected_script.get('status'))
print('selected_voiceover.json:', selected_voiceover_path, '| status:', selected_voiceover.get('status'))

if selected_script.get('status') != 'approved':
    print(f"UYARI: selected_script.json status='{selected_script.get('status')}' (approved degil)")
if selected_voiceover.get('status') != 'approved':
    print(f"UYARI: selected_voiceover.json status='{selected_voiceover.get('status')}' (approved degil)")
print('Not: onaylanmamis girdilerle render yine calisir (inceleme icin) ama nihai durum')
print('     review_required/failed kalir, ASLA approved/production_ready olmaz.')


## 2. Render yapılandırması ve onay bayrağı

Tüm render mantığı tek bir kanonik çağrıda (`render_utils.run_render_pipeline`) toplanmıştır: aralık kesme, altyazı hizalama (gerçek seslendirme sesinden `faster-whisper` ile; başarısız olursa orantılı fallback), dikey kadraj + altyazı yakma + ses miksleme (tek `ffmpeg` geçişi), teknik doğrulama ve onay kapısı değerlendirmesi.

In [ ]:
RENDER_MODE = 'blurred_background'  # 'blurred_background' veya 'center_crop'
TARGET_WIDTH = 1080
TARGET_HEIGHT = 1920
PREVIEW_MODE = True  # True = hizli/dusuk kalite onizleme; False = uretim kalitesi

DUCKING_ENABLED = True
ORIGINAL_AUDIO_BASE_VOLUME = 0.35
NARRATION_VOLUME = 1.0
DUCK_RATIO = 8

SUBTITLE_FONTSIZE = 20
SUBTITLE_MARGIN_V = 90
SUBTITLE_MAX_CHARS_PER_CUE = 42

SOURCE_VIDEO_PATH_OVERRIDE = ''  # zorla belirli bir ham kaynak video vermek istersen buraya yaz
print('Render modu:', RENDER_MODE, '| onizleme mi:', PREVIEW_MODE)
print("NOT: onay bu hucrede DEGIL, bolum 5'te (videoyu izledikten sonra) verilir.")


## 3. Render çalıştır (kanonik `render_utils.run_render_pipeline`)

In [ ]:
render_manifest = render_utils.run_render_pipeline(
    hot_moments_data, selected_script, selected_voiceover,
    hot_moments_path, selected_script_path, selected_voiceover_path,
    render_dir=RENDER_DIR,
    render_mode=RENDER_MODE, target_width=TARGET_WIDTH, target_height=TARGET_HEIGHT,
    preview_mode=PREVIEW_MODE, ducking_enabled=DUCKING_ENABLED,
    original_audio_base_volume=ORIGINAL_AUDIO_BASE_VOLUME, narration_volume=NARRATION_VOLUME,
    duck_ratio=DUCK_RATIO, subtitle_fontsize=SUBTITLE_FONTSIZE,
    subtitle_margin_v=SUBTITLE_MARGIN_V, subtitle_max_chars_per_cue=SUBTITLE_MAX_CHARS_PER_CUE,
    source_video_path_override=SOURCE_VIDEO_PATH_OVERRIDE,
)

print('Kaynak cozumlemesi:', render_manifest['inputs']['source_description'])
print('Altyazi kaynagi:', render_manifest['subtitles']['source'],
      f"({render_manifest['subtitles']['cue_count']} satir)")
print('Render dosyasi:', render_manifest['output']['final_video_path'],
      f"({render_manifest['output']['render_latency_sec']}s)")
print()
print('Teknik dogrulama (passed):', render_manifest['output']['validation']['passed'],
      '| probe backend:', render_manifest['output']['validation']['details']['probe_backend'])
print(json.dumps(render_manifest['output']['validation']['checks'], indent=2))
print()
print('Onay oncesi kosullar:', json.dumps(render_manifest['approval']['conditions'], indent=2))
if render_manifest['approval']['failed_conditions']:
    print('Saglanmayan kosul(lar):', render_manifest['approval']['failed_conditions'])
print('Render durumu:', render_manifest['status'], '-', render_manifest['status_reason'])


## 5. Render'ı izle ve ONAYLA (ayrı adım)

Videoyu **izledikten sonra** aşağıda yazdırılan SHA-256'yı `RENDER_REVIEWED_SHA256` alanına yapıştır.

Render onayı yalnızca **tüm** koşullar sağlandığında verilebilir: teknik doğrulama geçmiş **ve**
script onaylı **ve** seslendirme onaylı **ve** seslendirme-script hash eşleşmesi sağlanmış olmalı.
Herhangi biri eksikse onay reddedilir ve artefakt `review_required` kalır.

Sadece izlediğini kaydetmek istersen `RECORD_RENDER_REVIEW_ONLY = True` yap — bu `status`'ü değiştirmez.

In [ ]:
video_path = render_manifest['output']['final_video_path']
try:
    from IPython.display import Video, display as _display
    _display(Video(video_path, embed=True, width=320))
except Exception as exc:
    print('Video oynatici gosterilemedi:', exc)
    print('Dosyayi elle izle:', video_path)

render_review_info = approval.describe_for_review('render', render_manifest)
print('Icerik SHA-256:', render_review_info['content_sha256'])
if not render_review_info['reviewable']:
    print()
    print('!!! FINAL ONAY ENGELLI !!!')
    print(render_review_info['blocker'])
    for condition in render_review_info.get('blocking_conditions', []):
        print('  -', condition)

RENDER_REVIEWED_SHA256 = ''   # yukaridaki SHA-256'yi buraya yapistir
RENDER_APPROVED_BY = ''
RENDER_APPROVAL_NOTES = ''
RECORD_RENDER_REVIEW_ONLY = False  # True: sadece "izledim" kaydi, status DEGISMEZ

if not RENDER_REVIEWED_SHA256 or not RENDER_APPROVED_BY:
    print('Onay ATLANDI: RENDER_REVIEWED_SHA256 ve RENDER_APPROVED_BY doldurulmali.')
    print('status olarak kaliyor:', render_manifest['status'])
elif RECORD_RENDER_REVIEW_ONLY:
    try:
        render_manifest = approval.record_render_review(
            render_manifest, RENDER_REVIEWED_SHA256, RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        print('Inceleme kaydedildi (render_review). status DEGISMEDI:', render_manifest['status'])
    except ValueError as exc:
        print('INCELEME KAYDI REDDEDILDI:', exc)
else:
    try:
        render_manifest = approval.approve_artifact(
            render_manifest, 'render', RENDER_REVIEWED_SHA256,
            RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        print('Render ONAYLANDI (status:', render_manifest['status'], ')')
        print('NOT: onay yalnizca bir kayittir; bu notebook hicbir sey yayinlamaz.')
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('status olarak kaliyor:', render_manifest['status'])
        print("Sadece izledigini kaydetmek istersen RECORD_RENDER_REVIEW_ONLY = True yap.")


## 6. Dışa aktarım: `render_manifest.json`

In [ ]:
render_manifest_path = RENDER_DIR / 'render_manifest.json'
with open(render_manifest_path, 'w', encoding='utf-8') as f:
    json.dump(render_manifest, f, ensure_ascii=False, indent=2)
print('Kaydedildi:', render_manifest_path, '(status:', render_manifest['status'], ')')

render_validation_path = RENDER_DIR / 'render_validation.json'
with open(render_validation_path, 'w', encoding='utf-8') as f:
    json.dump(render_manifest['output']['validation'], f, ensure_ascii=False, indent=2)
print('Kaydedildi:', render_validation_path)

print('HATIRLATMA: bu notebook hicbir yere yukleme yapmadi (not_uploaded='
      f"{render_manifest['not_uploaded']}).")
print('Yayin, ayri ve elle onayli bir adimdir.')
